In [ ]:
#check GPU
!nvidia-smi

In [ ]:
#get current working directory
import os
os.getcwd()

In [ ]:
#print python version and location
import sys
print(sys.executable)
print(sys.version)

In [ ]:
#set environmental variables
os.environ["HF_HOME"] = "/path/to/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/path/to/.cache/huggingface/hub"

# offline mode
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

# disabling xet avoids the rust/reqwest path that often fails on clusters
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [ ]:
# Future imports 
from __future__ import annotations

# Standard library imports
import contextlib
import glob
import io
import itertools
import json
import logging
import os
import random
import re
import shutil
import time
import warnings
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

# Third-party imports
import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchmetrics
from albumentations.pytorch import ToTensorV2
from IPython.display import display
from matplotlib import patches
from matplotlib.lines import Line2D
from matplotlib.patches import ConnectionPatch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from PIL import Image, ImageDraw
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import (
    auc,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
from torchsummary import summary
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torchmetrics.classification import Accuracy, F1Score

from pytorch_grad_cam import (
    AblationCAM,
    EigenCAM,
    EigenGradCAM,
    FullGrad,
    GradCAM,
    GradCAMPlusPlus,
    HiResCAM,
    LayerCAM,
    ScoreCAM,
    XGradCAM,
)
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Optional / compatibility imports
try:
    import lightning.pytorch as pl
except Exception:  # pragma: no cover
    import pytorch_lightning as pl  # type: ignore

from pytorch_lightning.callbacks import Callback, EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

In [ ]:
# Python script to remove unwanted files and directories
# Define the path to the directory you want to clean
data_dir = "path/to/your/data/folder" #containing the train and test folders with normal and normal_not class-specific folders

# List of unwanted directories and files to look for
unwanted_directories = ['.ipynb_checkpoints']
unwanted_files = ['thumbs.db', 'Thumbs.db', '.DS_Store', 'DS_Store']

def clean_directory(directory):
    """
    Remove only the unwanted files (unwanted_files list)
    and unwanted directories (unwanted_directories list)
    in the given directory.
    """
    # Track the removed files and directories
    removed_items = []

    # Walk through the directory (bottom-up so we can remove subdirs cleanly)
    for root, dirs, files in os.walk(directory, topdown=False):
        # Remove unwanted files
        for file_name in files:
            file_path = os.path.join(root, file_name)
            if file_name in unwanted_files:
                print(f"Removing unwanted file: {file_path}")
                os.remove(file_path)
                removed_items.append(file_path)

        # Remove unwanted directories (using shutil.rmtree for non-empty dirs)
        for dir_name in dirs:
            if dir_name in unwanted_directories:
                dir_path = os.path.join(root, dir_name)
                print(f"Removing directory: {dir_path}")
                shutil.rmtree(dir_path)
                removed_items.append(dir_path)

    return removed_items

# Clean the data directory
print("Cleaning data_dir...")
removed_items = clean_directory(data_dir)

# Output summary of removed items
print(f"\nRemoved {len(removed_items)} items from data_dir.")
if not removed_items:
    print("No unwanted files or directories were found.")

In [ ]:
# Set Seeds for Full Reproducibility
def set_seed(seed):
    """
    Set the random seed
    """
    # 1. Python random
    random.seed(seed)
    # 2. NumPy
    np.random.seed(seed)
    # 3. PyTorch (CPU and GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # 4. Enable TF32 for float32 matmuls on A100 (max throughput, minimal precision loss)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('medium')
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(42)

In [ ]:
# Model Configuration dictionary; 5 fold cross validation, so we will save five models, each model with a single checkpoint with the lowest validation loss
h = {
    "model": "dpn68_new",
    "img_size": 224,
    "batch_size": 64,
    "num_workers": 2,
    "dropout": 0.3,
    "num_classes": 2,
    "pin_memory": True,
    "lr": 5e-5,
    "max_epochs": 64,
    "patience": 10,
    "balance": True,
    "n_splits": 5,
    "test_dir": "path/to/test/folder",
    "output_dir": "path/to/result/folder",
    "train_color": "blue",
    "val_color": "orange",
    "legend_loc": "best",
    "font_size": 12
    }

## Dataset + DataModule 

CXRDataset
Reads each CXR as grayscale (cv2.IMREAD_GRAYSCALE) and converts to 3-channel RGB (cv2.COLOR_GRAY2RGB) for ImageNet-pretrained backbones.
Applies Albumentations transforms (ToTensorV2) if provided.

Returns:
Train/Val: (image, label)
Test (return_paths=True): (image, label, filename) for case-level alignment.

DataModule (Lightning)
Binary classes: ["normal", "normal_not"] with mapping {normal:0, normal_not:1} (positive class = 1).

Transforms
Train: CLAHE (p=0.5) + Resize + Normalize + ToTensorV2
Val/Test: Resize + Normalize + ToTensorV2

Deterministic ordering
_make_split() sorts filenames (per-class + global) to guarantee stable ordering across folds/ensembles.

Imbalance handling
If h["balance"]=True, uses WeightedRandomSampler on train; otherwise standard shuffle.

Test set is always built
Reads from test_dir/<class>/ (fallback: test_dir/test/<class>/)
Exposes test_filenames and returns (img, label, filename) batches via test_dataloader().
Loader args are safe for num_workers=0 (no prefetch_factor / persistent_workers).

In [ ]:
# Dataset + DataModule
class CXRDataset(Dataset):
    """
    - Train/Val: returns (image, label)
    - Test:      returns (image, label, filename) when return_paths=True
    """
    def __init__(
        self,
        file_paths: List[Path],
        labels: List[int],
        transform: Optional[A.Compose] = None,
        return_paths: bool = False
    ) -> None:
        assert len(file_paths) == len(labels), "file_paths and labels must align"
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform
        self.return_paths = return_paths

    def __len__(self) -> int:
        return len(self.file_paths)

    def __getitem__(self, idx: int) -> Union[Tuple[torch.Tensor, int],
                                             Tuple[torch.Tensor, int, str]]:
        # always read as grayscale and expand to 3-channel RGB for pretrained backbones
        img = cv2.imread(str(self.file_paths[idx]), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Failed to read image: {self.file_paths[idx]}")
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        if self.transform is not None:
            img = self.transform(image=img)["image"]

        y = int(self.labels[idx])
        if self.return_paths:
            return img, y, self.file_paths[idx].name
        return img, y

class DataModule(pl.LightningDataModule):
    def __init__(self, h: Dict, fold_idx: Optional[int] = None) -> None:
        super().__init__()
        self.h = h
        self.fold_idx = fold_idx

        # Class definitions (index 1 == 'normal_not' == positive)
        self.class_names = ["normal", "normal_not"]
        self.class_to_idx = {n: i for i, n in enumerate(self.class_names)}
        print(f"[DataModule] Class to index mapping: {self.class_to_idx}")

        # ---- Transforms ----
        size = int(h["img_size"])
        # Train (with CLAHE)
        self.train_transform = A.Compose([
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
            A.Resize(size, size, interpolation=cv2.INTER_AREA),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
        # Val/Test (no augmentation)
        self.val_transform = A.Compose([
            A.Resize(size, size, interpolation=cv2.INTER_AREA),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
        self.test_transform = self.val_transform

        # DataLoader args (avoid persistent_workers when num_workers == 0)
        num_workers = int(h["num_workers"])
        self.loader_args = dict(
            batch_size=int(h["batch_size"]),
            num_workers=num_workers,
            pin_memory=bool(h["pin_memory"]),
            persistent_workers=bool(num_workers > 0),
            prefetch_factor=2 if num_workers > 0 else None,
            drop_last=False
        )
        # torch DataLoader disallows None prefetch_factor; remove key if None
        if self.loader_args["prefetch_factor"] is None:
            self.loader_args.pop("prefetch_factor")

        # Expose test filenames for downstream alignment (filled in setup)
        self.test_filenames: List[str] = []

    def _make_split(self, base: Path, split: str) -> Tuple[List[Path], List[int]]:
        """
        Reads images from either:
          - base/<split>/<class>/   when split is non-empty, or
          - base/<class>/           when split == "".
        Ensures a deterministic, filename-sorted order across classes.
        """
        fps, labs = [], []
        for cls in self.class_names:
            folder = (base / split / cls) if split else (base / cls)
            if not folder.exists():
                continue
            imgs: List[Path] = []
            for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"):
                imgs.extend(folder.glob(ext))
            # sort deterministically by filename
            imgs = sorted(imgs, key=lambda p: p.name.lower())
            fps.extend(imgs)
            labs.extend([self.class_to_idx[cls]] * len(imgs))
        # final global sort by filename to make cross-class order stable as well
        if fps:
            order = np.argsort([p.name.lower() for p in fps])
            fps = [fps[i] for i in order]
            labs = [labs[i] for i in order]
        return fps, labs

    def setup(self, stage: Optional[str] = None) -> None:
        # ---- Train/Val for cross-validation ----
        if self.fold_idx is not None:
            base = Path(self.h["output_dir"]) / f"fold_{self.fold_idx}"
            tr_fps, tr_labs = self._make_split(base, "train")
            vl_fps, vl_labs = self._make_split(base, "val")

            if self.h.get("balance", False) and len(tr_labs) > 0:
                counts = np.bincount(tr_labs, minlength=len(self.class_names))
                weights = (1.0 / np.maximum(counts, 1))[np.array(tr_labs)]
                weights = torch.as_tensor(weights, dtype=torch.double)
                self.train_sampler = WeightedRandomSampler(
                    weights, num_samples=len(weights), replacement=True
                )
                print("[DataModule] Using WeightedRandomSampler to correct class-imbalance")
            else:
                self.train_sampler = None
                print("[DataModule] Using standard shuffling (no imbalance correction)")

            self.train_dataset = CXRDataset(tr_fps, tr_labs, transform=self.train_transform, return_paths=False)
            self.val_dataset = CXRDataset(vl_fps, vl_labs, transform=self.val_transform,   return_paths=False)
        else:
            self.train_dataset = None
            self.val_dataset = None
            self.train_sampler = None
            print("[DataModule] Skipping train/val setup (fold_idx is None)")

        # ---- Always build TEST (shared across all ensemble members) ----
        td = Path(self.h["test_dir"])
        ts_fps, ts_labs = self._make_split(td, "")
        if not ts_fps:
            ts_fps, ts_labs = self._make_split(td, "test")
        self.test_dataset = CXRDataset(ts_fps, ts_labs, transform=self.test_transform, return_paths=True)
        self.test_filenames = [p.name for p in ts_fps]
        print(f"[DataModule] Test set: {len(ts_fps)} samples")

    def train_dataloader(self):
        if self.train_dataset is None:
            return None
        if self.train_sampler is not None:
            return DataLoader(
                self.train_dataset,
                sampler=self.train_sampler,
                shuffle=False,
                **self.loader_args
            )
        return DataLoader(
            self.train_dataset,
            shuffle=True,
            **self.loader_args
        )

    def val_dataloader(self):
        if self.val_dataset is None:
            return None
        return DataLoader(
            self.val_dataset,
            shuffle=False,
            **self.loader_args
        )

    def test_dataloader(self):
        # returns (image, label, filename) batches
        return DataLoader(
            self.test_dataset,
            shuffle=False,
            **self.loader_args
        )

## Model definition

1) We use DPN-68 from timm (dpn68.mx_in1k) with features_only=True.
Instead of returning logits directly, features_only=True returns a list of spatial feature maps at different stages. We take the final stage feature map (highest-level semantics) as the representation for classification and CAM-based explainability.
Input: image tensor of shape (B, 3, H, W)
Output from backbone: list of feature tensors, last one is (B, C, H′, W′)

2) CAM-friendly refinement: optional post 3×3 convolution
Many CAM/Grad-CAM style methods work best when the final feature map is produced by a 3×3 convolution.
To keep explanations stable across backbones and timm internals, the model automatically checks the last stage:
If the final stage is not written by a 3×3 Conv2d, we append a lightweight block:
Conv(3×3) → BatchNorm → SiLU
Otherwise, we leave the backbone output unchanged.
This makes the final feature map more consistent for explanation pipelines without changing the overall architecture dramatically.

3) Classification head
The head converts the final spatial feature map into class logits:
Global Average Pooling: (B, C, H′, W′) → (B, C, 1, 1)
Flatten: (B, C, 1, 1) → (B, C)
Dropout: regularization
Linear layer: (B, C) → (B, 2) for Normal vs TB

4) Label handling + loss (supports hard and soft targets)
The training code accepts:
Hard class indices: labels shaped (B,) (0/1)
One-hot or soft labels: shaped (B, 2) (useful for MixUp/CutMix/label smoothing)
Loss is computed as a negative log-likelihood over one-hot targets. This generalizes cross-entropy to soft targets cleanly.

5) Metrics
For binary classification (num_classes=2), we track:
Accuracy
F1 score
All metrics are logged at the epoch level (cleaner for medical reporting and plotting).

6) Optimizer + scheduler
We use:
AdamW (robust for modern CNN training)
CosineAnnealingLR across max_epochs (smooth LR decay)
This setup is stable for fine-tuning ImageNet-pretrained backbones on medical imaging datasets.

7) Notebook-friendly history tracking
The History callback collects epoch metrics into Python lists (train/val loss, accuracy, F1) so you can quickly plot learning curves without depending on external loggers.

In [ ]:
# Logging
LOGGER = logging.getLogger("tb_dpn68")
if not LOGGER.handlers:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    )
# Helpers to identify whether the final-stage feature map is written by a 3×3 Conv2d
class _ConvSpy:
    """
    Records forward-pass outputs of Conv2d modules to locate which convs wrote
    the final-stage feature map (same H×W as final output map).
    """

    def __init__(self) -> None:
        self.records: List[Tuple[str, nn.Module, Tuple[int, int, int, int]]] = []  # (name, mod, out.shape)
        self.hooks: List[torch.utils.hooks.RemovableHandle] = []

    def _hook(self, name: str, mod: nn.Conv2d):
        def _h(_m: nn.Module, _x: Tuple[torch.Tensor, ...], y: torch.Tensor) -> None:
            if isinstance(y, torch.Tensor) and y.ndim == 4:
                self.records.append((name, mod, tuple(y.shape)))  # type: ignore[arg-type]
        return _h

    def attach(self, module: nn.Module) -> None:
        for n, m in module.named_modules():
            if isinstance(m, nn.Conv2d):
                self.hooks.append(m.register_forward_hook(self._hook(n, m)))

    def detach(self) -> None:
        for h in self.hooks:
            try:
                h.remove()
            except Exception:
                pass
        self.hooks.clear()

@torch.no_grad()
def _discover_laststage_conv_k(
    backbone: nn.Module,
    img_size: int,
) -> Tuple[Optional[nn.Conv2d], Optional[Tuple[int, int]]]:
    """
    Returns:
        (last_conv_writing_final_stage_map, (kH, kW)) or (None, None) if not found.

    Notes:
    - Runs on CPU with a dummy tensor to avoid binding GPU during Lightning __init__.
    - For timm features_only backbones, forward returns a list/tuple of feature maps.
    """
    spy = _ConvSpy()
    spy.attach(backbone)

    x = torch.zeros(1, 3, img_size, img_size, dtype=torch.float32, device="cpu")
    feats = backbone(x)
    last = feats[-1] if isinstance(feats, (list, tuple)) else feats
    last_hw = tuple(last.shape[-2:])

    spy.detach()

    same_hw: List[Tuple[str, nn.Module, Tuple[int, int, int, int]]] = []
    for (n, m, shp) in spy.records:
        if tuple(shp[-2:]) == last_hw:
            same_hw.append((n, m, shp))

    if not same_hw:
        return None, None

    _, last_conv, _ = same_hw[-1]
    if not isinstance(last_conv, nn.Conv2d):
        return None, None

    k = last_conv.kernel_size
    k = k if isinstance(k, tuple) else (k, k)
    return last_conv, (int(k[0]), int(k[1]))

# Model: dpn68
class TBModel(pl.LightningModule):
    """
    Pediatric lateral CXR classifier (normal vs TB) using DPN-68 (timm) backbone.

    Expected hparams dict `h` (typical):
        h = {
            "model": "dpn68_new",
            "img_size": 224,
            "num_classes": 2,
            "dropout": 0.3,
            "lr": 5e-5,
            "weight_decay": 1e-4,
            "max_epochs": 64,
        }
    """

    def __init__(self, h: Dict[str, Union[str, int, float]]) -> None:
        super().__init__()
        self.h: Dict[str, Union[str, int, float]] = dict(h)
        self.save_hyperparameters({"h": self.h})

        name = str(self.h.get("model", "dpn68_new")).strip()
        if name != "dpn68_new":
            raise ValueError(f"Unsupported model '{name}'. This clean version supports ONLY 'dpn68_new'.")
        img_size = int(self.h.get("img_size", 224))
        drop_rate = float(self.h.get("dropout", 0.0))
        num_classes = int(self.h.get("num_classes", 2))
        self.num_classes = num_classes

        # Backbone: DPN-68 pretrained (ImageNet-1k)
        # features_only=True -> list of feature maps; take the last for classification and CAM.
        timm_id = "dpn68.mx_in1k"
        self.backbone = timm.create_model(
            timm_id,
            pretrained=True,
            features_only=True,
        )
        feat_dim = int(self.backbone.feature_info[-1]["num_chs"])

        # If the last stage isn't written by a 3×3 conv, add a 3×3 refinement conv for CAM stability.
        last_conv, k = _discover_laststage_conv_k(self.backbone, img_size=img_size)
        need_3x3 = (k is None) or (k != (3, 3))

        if need_3x3:
            LOGGER.info(
                "Backbone '%s': last-stage writer conv kernel=%s -> adding post 3×3 conv.",
                timm_id,
                str(k),
            )
            self.post3x3 = nn.Sequential(
                nn.Conv2d(feat_dim, feat_dim, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(feat_dim),
                nn.SiLU(inplace=True),
            )
        else:
            LOGGER.info("Backbone '%s': last-stage writer conv kernel=%s -> no post conv needed.", timm_id, str(k))
            self.post3x3 = nn.Identity()

        # Classification head: GAP → Dropout → Linear
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # (B,C,H,W) -> (B,C,1,1)
            nn.Flatten(),             # -> (B,C)
            nn.Dropout(drop_rate),
            nn.Linear(feat_dim, num_classes),
        )

        # Metrics (binary vs multiclass)
        if num_classes == 2:
            self.train_acc = Accuracy(task="binary")
            self.val_acc = Accuracy(task="binary")
            self.train_f1 = F1Score(task="binary")
            self.val_f1 = F1Score(task="binary")
        else:
            self.train_acc = Accuracy(task="multiclass", num_classes=num_classes, average="macro")
            self.val_acc = Accuracy(task="multiclass", num_classes=num_classes, average="macro")
            self.train_f1 = F1Score(task="multiclass", num_classes=num_classes, average="macro")
            self.val_f1 = F1Score(task="multiclass", num_classes=num_classes, average="macro")

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(x)
        feats = feats[-1] if isinstance(feats, (list, tuple)) else feats
        feats = self.post3x3(feats)
        return feats

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.forward_features(x)
        logits = self.head(feats)
        return logits

    @staticmethod
    def _nll_from_onehot_logits(logits: torch.Tensor, labels_oh: torch.Tensor) -> torch.Tensor:
        """
        NLL for (possibly soft) one-hot targets.
        This supports:
        - hard labels converted to one-hot
        - soft labels from MixUp/CutMix/label smoothing pipelines
        """
        logp = F.log_softmax(logits, dim=1)
        return -(labels_oh * logp).sum(dim=1).mean()

    def _prepare_targets(self, labels: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            labels_oh: (B, C) float
            tgt_idx:   (B,) long class indices (for metrics)
        Accepts:
            labels: (B,) integer class indices
                 or (B, C) one-hot / soft targets
        """
        if labels.ndim == 1:
            tgt_idx = labels.long()
            labels_oh = F.one_hot(tgt_idx, num_classes=self.num_classes).float()
        else:
            labels_oh = labels.float()
            tgt_idx = labels_oh.argmax(dim=1).long()
        return labels_oh, tgt_idx

    def training_step(self, batch: Tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
        imgs, labels = batch
        labels_oh, tgt_idx = self._prepare_targets(labels)

        logits = self(imgs)
        loss = self._nll_from_onehot_logits(logits, labels_oh)

        # Predictions for metrics
        pred_idx = logits.argmax(dim=1)
        if self.num_classes == 2:
            # torchmetrics binary expects {0,1}; pred_idx OK because logits argmax in {0,1}
            acc = self.train_acc(pred_idx, tgt_idx)
            f1 = self.train_f1(pred_idx, tgt_idx)
        else:
            acc = self.train_acc(pred_idx, tgt_idx)
            f1 = self.train_f1(pred_idx, tgt_idx)

        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("train_acc", acc,  on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("train_f1",  f1,   on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch: Tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
        imgs, labels = batch
        labels_oh, tgt_idx = self._prepare_targets(labels)

        logits = self(imgs)
        loss = self._nll_from_onehot_logits(logits, labels_oh)

        pred_idx = logits.argmax(dim=1)
        if self.num_classes == 2:
            acc = self.val_acc(pred_idx, tgt_idx)
            f1 = self.val_f1(pred_idx, tgt_idx)
        else:
            acc = self.val_acc(pred_idx, tgt_idx)
            f1 = self.val_f1(pred_idx, tgt_idx)

        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("val_acc",  acc,  on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log("val_f1",   f1,   on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def configure_optimizers(self):
        lr = float(self.h.get("lr", 5e-5))
        wd = float(self.h.get("weight_decay", 1e-4))
        max_epochs = int(self.h.get("max_epochs", 64))

        opt = torch.optim.AdamW(self.parameters(), lr=lr, weight_decay=wd)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

        return {"optimizer": opt, "lr_scheduler": {"scheduler": sch, "interval": "epoch"}}

# Lightweight history callback (epoch-level)
class History(pl.callbacks.Callback):
    """Collects a few epoch metrics into python lists for quick plotting."""

    def __init__(self) -> None:
        super().__init__()
        self.train_losses: List[float] = []
        self.val_losses: List[float] = []
        self.train_accs: List[float] = []
        self.val_accs: List[float] = []
        self.train_f1s: List[float] = []
        self.val_f1s: List[float] = []

    def on_train_epoch_end(self, trainer: "pl.Trainer", pl_module: "pl.LightningModule") -> None:
        m = trainer.callback_metrics
        if "train_loss" in m:
            self.train_losses.append(float(m["train_loss"]))
        if "train_acc" in m:
            self.train_accs.append(float(m["train_acc"]))
        if "train_f1" in m:
            self.train_f1s.append(float(m["train_f1"]))

    def on_validation_epoch_end(self, trainer: "pl.Trainer", pl_module: "pl.LightningModule") -> None:
        m = trainer.callback_metrics
        if "val_loss" in m:
            self.val_losses.append(float(m["val_loss"]))
        if "val_acc" in m:
            self.val_accs.append(float(m["val_acc"]))
        if "val_f1" in m:
            self.val_f1s.append(float(m["val_f1"]))

## Fine-tuning

1) For each fold in cross-validation (n_splits):
Instantiates a fold-specific DataModule(h, fold_idx=fold) that provides training/validation dataloaders.
Builds TBModel(h) configured only for dpn68_new.
Prints model and explainability sanity checks (parameter counts + final-stage conv writer probing).

Trains with:
EarlyStopping on val_loss (patience-controlled)
ModelCheckpoint saving the best model per fold by val_loss
A lightweight History callback to store epoch metrics for plotting
Saves training curves per fold and combined across folds.

2) Checkpointing layout and filenames
Checkpoints are saved to: {output_dir}/fold_{k}/dpn68_new/ with filenames that include epoch and validation metrics: dpn68_new_fold{k}_epochXX_valLoss_valAcc.ckpt
This keeps fold outputs fully isolated and easy to audit.

3) Why we probe “last-stage conv writer” (CAM sanity)
Explainability methods (e.g., EigenCAM/Grad-CAM) depend on the final spatial feature map.
Before training each fold, we:
Count total and trainable parameters (quick overfitting / compute sanity).
Report whether the model inserted a post 3×3 conv (post3x3) for CAM stability.
Probe the backbone Conv2d module that last writes the final-stage feature map, reporting its: module name, output tensor shape, kernel size
These checks make the model’s “last convolutional layer” behavior explicit, which prevents silent CAM regressions when swapping timm weights or upgrading packages.

4) Trainer configuration
The trainer is configured for modern fine-tuning:
accelerator="gpu", devices=1 (adjustable via h)
precision="bf16-mixed" (efficient on Ampere+ GPUs; configurable)
max_epochs, log_every_n_steps controlled via h
All key knobs (epochs, patience, precision, etc.) are sourced from h, so the notebook remains fold- and experiment-reproducible.

5) Outputs we get After running:
Per-fold: best checkpoint (.ckpt); training_curve.png
Across folds: combined_curves_dpn68_new.png under {output_dir}/dpn68_new/

In [ ]:
# Logging
LOGGER = logging.getLogger("tb_dpn68_ft")
if not LOGGER.handlers:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    )

# Output suppression: hide DataModule's misleading prints, then print our own
@contextlib.contextmanager
def suppress_output(enabled: bool = True):
    if not enabled:
        yield
        return
    old_out, old_err = os.sys.stdout, os.sys.stderr
    try:
        os.sys.stdout = io.StringIO()
        os.sys.stderr = io.StringIO()
        yield
    finally:
        os.sys.stdout = old_out
        os.sys.stderr = old_err

# Checkpoint callback (dpn68)
def get_model_checkpoint_callback(h: Dict[str, Any], fold: int) -> ModelCheckpoint:
    model_name = str(h.get("model", "dpn68_new")).strip()
    if model_name != "dpn68_new":
        raise ValueError("This fine-tuning block supports ONLY h['model'] == 'dpn68_new'.")

    ckpt_dir = os.path.join(str(h["output_dir"]), f"fold_{fold}", model_name)
    os.makedirs(ckpt_dir, exist_ok=True)

    filename_template = (
        f"{model_name}_fold{fold}_"
        "epoch{epoch:02d}_"
        "{val_loss:.4f}_{val_acc:.4f}"
    )

    return ModelCheckpoint(
        dirpath=ckpt_dir,
        filename=filename_template,
        save_top_k=1,
        verbose=True,
        monitor="val_loss",
        mode="min",
        save_last=False,
        save_weights_only=False,
        auto_insert_metric_name=False,
    )

# Plotting helpers
def plot_training_curves(
    train_losses: List[float],
    val_losses: List[float],
    train_accs: Optional[List[float]] = None,
    val_accs: Optional[List[float]] = None,
    save_path: Optional[Union[str, Path]] = None,
    figsize: Tuple[int, int] = (8, 6),
    train_color: str = "blue",
    val_color: str = "orange",
    alpha: float = 0.85,
    xlabel_font_size: int = 26,
    ylabel_font_size: int = 26,
    title_font_size: int = 26,
    legend_loc: str = "best",
    font_size: int = 26,
) -> None:
    train_epochs = range(1, len(train_losses) + 1)
    val_epochs = range(1, len(val_losses) + 1)

    plt.figure(figsize=figsize)
    plt.plot(train_epochs, train_losses, label="Train Loss", color=train_color, alpha=alpha, linewidth=2)
    plt.plot(val_epochs, val_losses, label="Val Loss", color=val_color, alpha=alpha, linewidth=2)

    if train_accs is not None and val_accs is not None:
        te = range(1, len(train_accs) + 1)
        ve = range(1, len(val_accs) + 1)
        plt.plot(te, train_accs, label="Train Acc", linestyle="--", color=train_color, linewidth=2)
        plt.plot(ve, val_accs, label="Val Acc", linestyle="--", color=val_color, linewidth=2)

    plt.xlabel("Epoch", fontsize=xlabel_font_size)
    plt.ylabel("Value", fontsize=ylabel_font_size)
    plt.title("Training Curves", fontsize=title_font_size)
    plt.legend(loc=legend_loc, fontsize=font_size)
    plt.grid(True)
    plt.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=400)
        plt.close()
    else:
        plt.show()

def plot_all_curves(h: Dict[str, Any], histories: List["History"]) -> None:
    model_name = str(h.get("model", "dpn68_new"))
    out_dir = Path(str(h["output_dir"]))

    for fold_idx, hist in enumerate(histories):
        model_dir = out_dir / f"fold_{fold_idx}" / model_name
        model_dir.mkdir(parents=True, exist_ok=True)
        save_path = model_dir / "training_curve.png"

        plot_training_curves(
            train_losses=hist.train_losses,
            val_losses=hist.val_losses,
            train_accs=hist.train_accs,
            val_accs=hist.val_accs,
            save_path=save_path,
            figsize=(8, 6),
            train_color=str(h.get("train_color", "blue")),
            val_color=str(h.get("val_color", "orange")),
            alpha=float(h.get("alpha", 0.8)),
            xlabel_font_size=int(h.get("xlabel_font_size", 12)),
            ylabel_font_size=int(h.get("ylabel_font_size", 12)),
            title_font_size=int(h.get("title_font_size", 14)),
            legend_loc=str(h.get("legend_loc", "best")),
            font_size=int(h.get("font_size", 12)),
        )
        print(f"[INFO] Saved fold-{fold_idx} curves → {save_path}")

def plot_combined_curves(h: Dict[str, Any], histories: List["History"]) -> None:
    model_name = str(h.get("model", "dpn68_new"))
    out_dir = Path(str(h["output_dir"]))
    model_root = out_dir / model_name
    model_root.mkdir(parents=True, exist_ok=True)
    combined_path = model_root / f"combined_curves_{model_name}.png"

    fig, ax = plt.subplots(figsize=(10, 7))
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for fold_idx, hist in enumerate(histories):
        te = range(1, len(hist.train_losses) + 1)
        ve = range(1, len(hist.val_losses) + 1)
        ax.plot(te, hist.train_losses, label=f"fold-{fold_idx} train", color=colors[fold_idx % len(colors)], linestyle="-")
        ax.plot(ve, hist.val_losses, label=f"fold-{fold_idx} val",   color=colors[fold_idx % len(colors)], linestyle="--")

    ax.set_xlabel("Epoch", fontsize=int(h.get("xlabel_font_size", 12)))
    ax.set_ylabel("Loss", fontsize=int(h.get("ylabel_font_size", 12)))
    ax.set_title(f"Combined Training & Validation Curves ({model_name})", fontsize=int(h.get("title_font_size", 14)))
    ax.legend(loc="upper right", fontsize=int(h.get("font_size", 12)))
    plt.tight_layout()
    plt.savefig(combined_path, dpi=300)
    plt.close(fig)
    print(f"[INFO] Saved combined curves → {combined_path}")

# Model/probe utilities (dpn68_new + CAM sanity) — used ONCE (fold 0)
def _format_param_count(n: int) -> str:
    if n >= 1_000_000:
        return f"{n / 1_000_000:.2f}M"
    if n >= 1_000:
        return f"{n / 1_000:.2f}K"
    return str(n)

def _count_params(model: nn.Module) -> Tuple[int, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

@torch.no_grad()
def _probe_backbone_last_writer(
    backbone: nn.Module,
    img_size: int,
) -> Tuple[str, Tuple[int, int, int, int], Tuple[int, int]]:
    records: List[Tuple[str, nn.Module, Tuple[int, int, int, int]]] = []
    hooks: List[torch.utils.hooks.RemovableHandle] = []

    def _make_hook(name: str, m: nn.Conv2d):
        def _h(_m: nn.Module, _in: Tuple[torch.Tensor, ...], out: torch.Tensor) -> None:
            if isinstance(out, torch.Tensor) and out.ndim == 4:
                records.append((name, m, tuple(out.shape)))  # type: ignore[arg-type]
        return _h

    for n, m in backbone.named_modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(_make_hook(n, m)))

    x = torch.zeros(1, 3, img_size, img_size, dtype=torch.float32, device="cpu")
    feats = backbone(x)
    last = feats[-1] if isinstance(feats, (list, tuple)) else feats
    last_hw = tuple(last.shape[-2:])

    for h in hooks:
        try:
            h.remove()
        except Exception:
            pass

    same_hw = [(n, m, shp) for (n, m, shp) in records if tuple(shp[-2:]) == last_hw]
    if not same_hw:
        raise RuntimeError("No Conv2d writer found for the final stage feature map.")

    name, mod, shp = same_hw[-1]
    assert isinstance(mod, nn.Conv2d)
    k = mod.kernel_size if isinstance(mod.kernel_size, tuple) else (mod.kernel_size, mod.kernel_size)
    k = (int(k[0]), int(k[1]))
    return name, shp, k

def _summarize_inserted_3x3(model: "TBModel") -> Tuple[bool, int, Tuple[int, int, int, int]]:
    post = getattr(model, "post3x3", None)
    if isinstance(post, nn.Sequential) and len(post) > 0 and isinstance(post[0], nn.Conv2d):
        conv = post[0]
        bn = post[1] if len(post) > 1 and isinstance(post[1], nn.BatchNorm2d) else None
        cout, cin, kh, kw = tuple(conv.weight.shape)

        p_conv = conv.weight.numel() + (conv.bias.numel() if conv.bias is not None else 0)
        p_bn = (bn.weight.numel() + bn.bias.numel()) if bn is not None else 0
        return True, int(p_conv + p_bn), (cout, cin, kh, kw)

    return False, 0, (0, 0, 0, 0)

# Data summary helpers — print *correct* class-wise counts (train/val/test)
def _extract_targets_from_dataset(ds: Any) -> Optional[List[int]]:
    """
    Best-effort extraction of integer class targets from common dataset types:
    - torchvision ImageFolder: ds.targets or ds.samples
    - torch.utils.data.Subset wrapping ImageFolder
    """
    if isinstance(ds, Subset):
        base = ds.dataset
        base_targets = _extract_targets_from_dataset(base)
        if base_targets is None:
            return None
        return [int(base_targets[i]) for i in ds.indices]

    # torchvision ImageFolder commonly exposes `targets` (list[int])
    if hasattr(ds, "targets"):
        t = getattr(ds, "targets")
        try:
            return [int(x) for x in list(t)]
        except Exception:
            pass

    # torchvision ImageFolder also has `samples`: List[Tuple[path, class_idx]]
    if hasattr(ds, "samples"):
        s = getattr(ds, "samples")
        try:
            return [int(y) for (_, y) in list(s)]
        except Exception:
            pass

    # some custom datasets use `labels`
    if hasattr(ds, "labels"):
        lab = getattr(ds, "labels")
        try:
            return [int(x) for x in list(lab)]
        except Exception:
            pass

    return None

def _infer_class_names(ds: Any, fallback: Tuple[str, ...] = ("normal", "normal_not")) -> List[str]:
    if hasattr(ds, "classes"):
        try:
            return [str(c) for c in list(getattr(ds, "classes"))]
        except Exception:
            pass
    return list(fallback)

def _get_split_dataset(dm: Any, split: str) -> Optional[Any]:
    """
    Tries:
      1) dm.<split>_dataset
      2) dm.<split>_dataloader().dataset
    split in {"train","val","test"}
    """
    ds_attr = f"{split}_dataset"
    if hasattr(dm, ds_attr):
        return getattr(dm, ds_attr)

    dl_fn = f"{split}_dataloader"
    if hasattr(dm, dl_fn):
        dl = getattr(dm, dl_fn)()
        if hasattr(dl, "dataset"):
            return dl.dataset

    return None

# Training loop (CV) — dpn68_new end-to-end fine-tuning
def train_cv(
    h: Dict[str, Any],
    *,
    suppress_datamodule_prints: bool = True,
    print_model_once: bool = True,
) -> List["History"]:
    """
    Cross-validated end-to-end fine-tuning on fold_i train/val splits.

    Fixes:
    - prints model architecture + CAM checks only once (fold 0) if print_model_once=True
    - suppresses DataModule's own prints (optional) and prints corrected class-wise split counts
    """
    model_name = str(h.get("model", "dpn68_new")).strip()
    if model_name != "dpn68_new":
        raise ValueError("This training block supports ONLY h['model'] == 'dpn68_new'.")

    out_dir = Path(str(h["output_dir"]))
    out_dir.mkdir(parents=True, exist_ok=True)

    n_splits = int(h.get("n_splits", 5))
    max_epochs = int(h.get("max_epochs", 64))
    patience = int(h.get("patience", 10))
    img_size = int(h.get("img_size", 224))

    # keep prior defaults unless adding these keys to h
    accelerator = str(h.get("accelerator", "gpu"))
    devices = int(h.get("devices", 1))
    precision = str(h.get("precision", "bf16-mixed"))
    log_every_n_steps = int(h.get("log_every_n_steps", 30))

    histories: List["History"] = []

    for fold in range(n_splits):
        print(f"\n==== Starting training for Fold {fold} ({model_name}) ====")

        # Data
        dm = DataModule(h, fold_idx=fold)  # provided by you elsewhere        
        with suppress_output(enabled=suppress_datamodule_prints):
            dm.setup()
        # Model
        model = TBModel(h)
        # Print model architecture + CAM sanity checks only once (fold 0)
        if print_model_once and fold == 0:
            print("\n-----------------------------------")
            print("[Model] Architecture (printed once):")
            print(model)

            total_p, train_p = _count_params(model)
            print(f"[Model] Params: total={_format_param_count(total_p)} | trainable={_format_param_count(train_p)}")

            inserted, pcount, wshape = _summarize_inserted_3x3(model)
            if inserted:
                print(f"[Model] post3x3 inserted: weight_shape={wshape} | extra_params={_format_param_count(pcount)}")
            else:
                print("[Model] post3x3 not inserted (backbone already ends with suitable last-stage 3×3 writer).")

            try:
                name, shp, k = _probe_backbone_last_writer(model.backbone, img_size=img_size)
                print(f"[Model] Backbone last-stage writer: {name} | out={shp} | kernel={k}")
            except Exception as e:
                print(f"[Model] WARNING: could not probe backbone last writer ({type(e).__name__}: {e})")
            print("-----------------------------------\n")

        # Callbacks
        hist_cb = History()
        es_cb = EarlyStopping(monitor="val_loss", patience=patience, mode="min", verbose=True)
        ckpt_cb = get_model_checkpoint_callback(h, fold)

        # Trainer
        trainer = pl.Trainer(
            accelerator=accelerator,
            devices=devices,
            precision=precision,
            max_epochs=max_epochs,
            default_root_dir=str(out_dir),
            callbacks=[es_cb, ckpt_cb, hist_cb],
            log_every_n_steps=log_every_n_steps,
            enable_progress_bar=True,
            enable_checkpointing=True,
        )

        trainer.fit(model, datamodule=dm)
        histories.append(hist_cb)
        del model
        torch.cuda.empty_cache()
    return histories

# Run
histories = train_cv(h, suppress_datamodule_prints=True, print_model_once=True)
plot_all_curves(h, histories)
plot_combined_curves(h, histories)

## Validation inference

1) Fold-wise checkpoint selection (best val_loss)
For each fold k, checkpoints are expected under: {output_dir}/fold_{k}/dpn68_new/, with filenames formatted as: dpn68_new_fold{k}_epochXX_{val_loss}_{val_acc}.ckpt
The script parses val_loss from the filename and selects the checkpoint with the lowest validation loss.

2) Validation dataset reconstruction from physical folders
To keep inference independent of training internals, we build a validation-only DataLoader from the physical split folders:
{output_dir}/fold_{k}/val/normal
{output_dir}/fold_{k}/val/normal_not (preferred)
(If a legacy split used val/tb, the code will automatically fall back to that directory.)
Labels follow the study convention:
0 = normal
1 = normal_not (TB)
The code prints fold-wise validation support:
normal=N0, normal_not=N1, total=N0+N1

3) Preprocessing and inference precision
Images are preprocessed using the same ImageNet normalization used during fine-tuning:
resize to h["img_size"]=224
normalize with mean/std = (0.485, 0.456, 0.406) / (0.229, 0.224, 0.225)
Inference uses bf16 autocast on GPU for throughput, but casts logits back to fp32 before softmax. This avoids numerical edge cases in probability computation while keeping inference efficient.

4) Metrics and confusion matrix (per fold)
For each fold, we compute:
Balanced Accuracy
Sensitivity (Recall for TB / normal_not)
Specificity
Precision
NPV
Macro F1
MCC
Cohen’s κ
Youden’s J
ROC AUC
We also compute a 2×2 confusion matrix and save:
confusion_fold{k}.png
metrics_fold{k}.csv
into: {output_dir}/fold_{k}/dpn68_new/validation_inference/

5) Cross-fold aggregation + ROC visualization
After processing all folds, the script:
saves all_folds_validation_metrics.csv under {output_dir}/dpn68_new/
prints mean ± std across folds for each metric

plots:
ROC curve per fold; mean ROC curve (interpolated on a common FPR grid)
The ROC summary plot is saved as: {output_dir}/dpn68_new/roc_validation_summary.png

In [ ]:
# Confusion matrix helper
def plot_confusion_pred_true(
    cm_pred_true: np.ndarray,
    classes: List[str],
    save_path: str | Path,
    figsize: Tuple[int, int] = (8, 6),
    cmap: str = "Greens",
    alpha: float = 0.85,
    xt_label_rotation: int = 0,
    xt_label_size: int = 22,
    yt_label_size: int = 22,
    xlabel_font_size: int = 26,
    ylabel_font_size: int = 26,
    text_color: str = "black",
    text_font_size: int = 38,
) -> None:
    """
    Plot confusion matrix with:
      - rows = predicted labels
      - cols = true labels
    Label order must match `classes` and encoding:
      classes[0] = "Normal" (label 0)
      classes[1] = "TB"     (label 1)

    Expected layout:
      cm[0,0]=TN, cm[0,1]=FN
      cm[1,0]=FP, cm[1,1]=TP
    """
    cm = np.asarray(cm_pred_true, dtype=int)
    assert cm.shape == (2, 2), f"Expected (2,2), got {cm.shape}"

    fig, ax = plt.subplots(figsize=figsize)
    cax = ax.matshow(cm, cmap=cmap, alpha=alpha)
    fig.colorbar(cax)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(classes, rotation=xt_label_rotation, fontsize=xt_label_size)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(classes, fontsize=yt_label_size)

    # xlabel bottom, ylabel left
    ax.set_xlabel("True Label", fontsize=xlabel_font_size)
    ax.set_ylabel("Predicted Label", fontsize=ylabel_font_size)

    # Annotate cells
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, int(v), ha="center", va="center",
                color=text_color, fontsize=text_font_size)

    plt.tight_layout()
    save_path = str(save_path)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

# Helpers — checkpoint selection + folder discovery
def _find_best_ckpt_by_val_loss(fold_base: Union[str, Path], fold: int, model_name: str) -> Optional[Tuple[str, float]]:
    """
    Expects checkpoint filenames like:
      dpn68_new_fold{fold}_epochXX_{val_loss:.4f}_{val_acc:.4f}.ckpt

    Returns (best_ckpt_path, best_val_loss) or None.
    """
    fold_base = str(fold_base)
    pattern = os.path.join(fold_base, f"{model_name}_fold{fold}_epoch*_*.ckpt")
    ckpts = sorted(glob.glob(pattern))
    if not ckpts:
        return None

    best_loss = float("inf")
    best_ckpt: Optional[str] = None

    # extract val_loss from filename (first float after epoch)
    rx = re.compile(rf".*_fold{fold}_epoch\d+_(\d+\.\d+)_\d+\.\d+\.ckpt$")
    for p in ckpts:
        m = rx.match(os.path.basename(p))
        if not m:
            continue
        loss_val = float(m.group(1))
        if loss_val < best_loss:
            best_loss, best_ckpt = loss_val, p

    if best_ckpt is None:
        return None
    return best_ckpt, best_loss

def _existing_val_class_dirs(output_dir: str, fold: int) -> Tuple[Optional[str], Optional[str]]:
    base = os.path.join(output_dir, f"fold_{fold}", "val")

    normal_dir = os.path.join(base, "normal")
    # prefer 'normal_not', but fallback to 'tb' if that was used historically
    abnormal_dir_candidates = [os.path.join(base, "normal_not"), os.path.join(base, "tb")]

    if not os.path.isdir(normal_dir):
        normal_dir = None

    abnormal_dir = None
    for c in abnormal_dir_candidates:
        if os.path.isdir(c):
            abnormal_dir = c
            break

    return normal_dir, abnormal_dir

def _list_images(folder: str) -> List[str]:
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
    paths = []
    for e in exts:
        paths.extend(glob.glob(os.path.join(folder, f"*{e}")))
        paths.extend(glob.glob(os.path.join(folder, f"*{e.upper()}")))
    return sorted(set(paths))

# Main validation inference
def inference_validation_with_curves(h: Dict[str, Any]) -> None:
    # enforce dpn68_new-only
    model_name = str(h.get("model", "dpn68_new")).strip()
    if model_name != "dpn68_new":
        raise ValueError("This inference block supports ONLY h['model'] == 'dpn68_new'.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    all_fold_records: List[Dict[str, Any]] = []
    roc_data: List[Dict[str, Any]] = []

    # plotting style defaults (honor h)
    fold_colors = ["red", "blue", "green", "orange", "brown"]
    fold_curve_alpha = 0.9
    fold_curve_lw = 2
    mean_line_color = "black"
    mean_line_lw = 2
    axis_fontsize = int(h.get("xlabel_font_size", 12))

    for fold in range(int(h.get("n_splits", 5))):
        fold_base = os.path.join(str(h["output_dir"]), f"fold_{fold}", model_name)
        if not os.path.isdir(fold_base):
            print(f"⚠️ Skipping fold {fold}: folder not found → {fold_base}")
            continue

        best = _find_best_ckpt_by_val_loss(fold_base=fold_base, fold=fold, model_name=model_name)
        if best is None:
            print(f"⚠️ Skipping fold {fold}: no valid checkpoints in {fold_base}")
            continue
        best_ckpt, best_loss = best

        print(f"\n🎯 Fold {fold} validation inference")
        print(f" → Best checkpoint: {os.path.basename(best_ckpt)} (val_loss={best_loss:.4f})")

        # Build val set from physical directories
        normal_dir, abnormal_dir = _existing_val_class_dirs(str(h["output_dir"]), fold=fold)
        if normal_dir is None or abnormal_dir is None:
            print(f"⚠️ Fold {fold}: missing val class directories under {os.path.join(str(h['output_dir']), f'fold_{fold}', 'val')}")
            continue

        normal_paths = _list_images(normal_dir)
        abnormal_paths = _list_images(abnormal_dir)

        # IMPORTANT: match your study naming: normal vs normal_not
        # Label convention: 0 = normal, 1 = normal_not (TB)
        if (not normal_paths) and (not abnormal_paths):
            print(f"⚠️ Fold {fold}: no validation images found.")
            continue

        print(f" [Fold {fold}] Val counts: normal={len(normal_paths)} | normal_not={len(abnormal_paths)} | total={len(normal_paths)+len(abnormal_paths)}")

        val_paths = np.array(normal_paths + abnormal_paths)
        val_labels = np.array([0] * len(normal_paths) + [1] * len(abnormal_paths), dtype=np.int64)

        # Same core normalization as training (ImageNet)
        val_transform = A.Compose(
            [
                A.Resize(int(h["img_size"]), int(h["img_size"]), interpolation=cv2.INTER_AREA),
                A.Normalize(
                    mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225),
                ),
                ToTensorV2(),
            ]
        )
        val_dataset = CXRDataset(val_paths, val_labels, transform=val_transform)
        val_loader = torch.utils.data.DataLoader(
            val_dataset,
            batch_size=int(h["batch_size"]),
            shuffle=False,
            num_workers=int(h.get("num_workers", 2)),
            pin_memory=bool(h.get("pin_memory", True)),
            persistent_workers=False,
            prefetch_factor=2,
            drop_last=False,
        )
        # Load model (dpn68_new-only TBModel) from checkpoint
        model = TBModel.load_from_checkpoint(best_ckpt, h=h)
        model.to(device)
        model.eval()
        y_true: List[int] = []
        y_pred: List[int] = []
        y_score: List[float] = []
        use_amp = (device.type == "cuda")
        amp_ctx = torch.cuda.amp.autocast if use_amp else torch.cpu.amp.autocast

        with torch.inference_mode():
            for imgs, labels in val_loader:
                imgs = imgs.to(device, non_blocking=True)

                # bf16 forward, then cast logits to fp32 for softmax stability
                with amp_ctx(dtype=torch.bfloat16):
                    logits = model(imgs)

                logits_fp32 = logits.float()
                probs_tb = torch.softmax(logits_fp32, dim=1)[:, 1].detach().cpu().numpy()
                preds = logits_fp32.argmax(dim=1).detach().cpu().numpy()

                y_score.extend(probs_tb.tolist())
                y_pred.extend(preds.tolist())
                y_true.extend(labels.numpy().tolist())

        y_true_np = np.asarray(y_true, dtype=np.int64)
        y_pred_np = np.asarray(y_pred, dtype=np.int64)
        y_score_np = np.asarray(y_score, dtype=np.float64)

        # Confusion
        TP = int(np.sum((y_true_np == 1) & (y_pred_np == 1)))
        FP = int(np.sum((y_true_np == 0) & (y_pred_np == 1)))
        FN = int(np.sum((y_true_np == 1) & (y_pred_np == 0)))
        TN = int(np.sum((y_true_np == 0) & (y_pred_np == 0)))
        cm_pred_true = np.array([[TP, FP], [FN, TN]], dtype=int)

        # Scalar metrics
        bal_acc = float(balanced_accuracy_score(y_true_np, y_pred_np))
        sensitivity = float(recall_score(y_true_np, y_pred_np, pos_label=1))
        specificity = float(TN / (TN + FP)) if (TN + FP) > 0 else 0.0
        precision = float(precision_score(y_true_np, y_pred_np, pos_label=1, zero_division=0))
        npv = float(TN / (TN + FN)) if (TN + FN) > 0 else 0.0
        fscore = float(f1_score(y_true_np, y_pred_np, average="macro"))
        mcc = float(matthews_corrcoef(y_true_np, y_pred_np))
        kappa = float(cohen_kappa_score(y_true_np, y_pred_np))
        youden_j = float(sensitivity + specificity - 1.0)
        print(f" ▶️ Fold {fold} Val: BalAcc={bal_acc:.4f}  Sens={sensitivity:.4f}  Spec={specificity:.4f}")

        # ROC
        fpr, tpr, _ = roc_curve(y_true_np, y_score_np, pos_label=1)
        roc_auc_val = float(auc(fpr, tpr))
        roc_data.append({"fold": fold, "fpr": fpr, "tpr": tpr, "auc": roc_auc_val})

        # Fold output folder
        valinf_root = os.path.join(fold_base, "validation_inference")
        os.makedirs(valinf_root, exist_ok=True)

        # Confusion plot
        cm_path = os.path.join(valinf_root, f"confusion_fold{fold}.png")
        plot_confusion_pred_true(
            cm_pred_true=cm_pred_true,
            classes=["Normal", "TB"],
            save_path=cm_path,
            figsize=(8, 6),
        )
        print(f"   [INFO] Saved confusion matrix → {cm_path}")

        record = {
            "fold": fold,
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "TN": TN,
            "balanced_accuracy": bal_acc,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "precision": precision,
            "NPV": npv,
            "F_score": fscore,
            "MCC": mcc,
            "Cohen_Kappa": kappa,
            "Youden_J": youden_j,
            "roc_auc": roc_auc_val,
        }
        all_fold_records.append(record)

        metrics_path = os.path.join(valinf_root, f"metrics_fold{fold}.csv")
        pd.DataFrame([record]).to_csv(metrics_path, index=False)
        print(f" [INFO] Saved per-fold validation metrics → {metrics_path}")
        print(f" ▶️ Fold {fold} AUC={roc_auc_val:.4f}")

        del model
        torch.cuda.empty_cache()

    if not all_fold_records:
        print("⚠️ No validation metrics saved (no folds processed).")
        return

    # Combined root under {output_dir}/dpn68_new/
    combined_root = os.path.join(str(h["output_dir"]), model_name)
    os.makedirs(combined_root, exist_ok=True)

    all_out_path = os.path.join(combined_root, "all_folds_validation_metrics.csv")
    pd.DataFrame(all_fold_records).to_csv(all_out_path, index=False)
    print(f"\n✅ Saved all-fold validation metrics → {all_out_path}")

    # Cross-fold summary
    stats = (
        pd.DataFrame(all_fold_records)
        .drop(columns=["fold", "TP", "FP", "FN", "TN"])
        .agg(["mean", "std"])
    )

    display_map = {
        "balanced_accuracy": "balanced_accuracy",
        "sensitivity": "sensitivity",
        "specificity": "specificity",
        "precision": "precision",
        "NPV": "NPV",
        "F_score": "F_score",
        "MCC": "MCC",
        "Cohen_Kappa": "Cohen_Kappa",
        "Youden_J": "Youden_J",
        "roc_auc": "ROC_AUC",
    }

    print("\nCross-Fold Summary Statistics 📊")
    for col in stats.columns:
        disp_name = display_map.get(col, col)
        mu = float(stats.loc["mean", col])
        sd = float(stats.loc["std", col])
        print(f" • {disp_name}: {mu:.4f} ± {sd:.4f}")

    # Combined ROC curves + mean ROC
    mean_fpr = np.linspace(0.0, 1.0, 200)
    interp_tprs: List[np.ndarray] = []

    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
    for idx, rd in enumerate(roc_data):
        ax_roc.plot(
            rd["fpr"],
            rd["tpr"],
            color=fold_colors[idx % len(fold_colors)],
            alpha=fold_curve_alpha,
            lw=fold_curve_lw,
            label=f"fold-{rd['fold']}: AUC={rd['auc']:.4f}",
        )

        itpr = np.interp(mean_fpr, rd["fpr"], rd["tpr"])
        itpr[0] = 0.0
        interp_tprs.append(itpr)

    mean_tpr = np.mean(np.vstack(interp_tprs), axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = float(auc(mean_fpr, mean_tpr))

    ax_roc.plot(
        mean_fpr,
        mean_tpr,
        color=mean_line_color,
        lw=mean_line_lw,
        linestyle="--",
        label=f"Mean ROC: AUC={mean_auc:.4f}",
    )

    ax_roc.set_xlim([0.0, 1.0])
    ax_roc.set_ylim([0.0, 1.02])
    ax_roc.set_xlabel("False Positive Rate", fontsize=axis_fontsize)
    ax_roc.set_ylabel("True Positive Rate", fontsize=axis_fontsize)
    ax_roc.legend(loc=str(h.get("legend_loc", "best")), fontsize=int(h.get("font_size", 12)))
    ax_roc.grid(False)

    
    plt.tight_layout()
    roc_fig_path = os.path.join(combined_root, "roc_validation_summary.png")
    fig_roc.savefig(roc_fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"\n✅ Saved Validation ROC curves plot → {roc_fig_path}")
    print("\n🎉 Validation inference + ROC plotting complete!")

# Run validation inference
inference_validation_with_curves(h)

## Test-time evaluation

1) Test loader construction (study DataModule)
We instantiate DataModule(h, fold_idx=None) and call setup(). This skips train/val construction and builds the test dataset from h["test_dir"]. The test dataloader yields:
imgs: (B, 3, H, W) tensors (CXR read in grayscale → converted to RGB)
labels: integer class indices with 0=normal, 1=tb
Builds one test DataLoader using a direct dataset that reads grayscale CXR images, expands to 3-channel RGB, applies ImageNet normalization, and returns (image, label, filename).
For each fold k, selects the checkpoint in output_dir/fold_k/dpn68_new/ with the lowest validation loss embedded in the filename.
Runs forward passes under CUDA autocast (bf16 if supported, else fp16), casts logits to fp32, and computes:

2) Fold-wise checkpoint selection (best val_loss)
For each fold k, we search checkpoints under: {output_dir}/fold_{k}/dpn68_new/ and pick the checkpoint with minimum validation loss using the filename template:
dpn68_new_fold{k}_epochXX_{val_loss:.4f}_{val_acc:.4f}.ckpt
This ensures we evaluate the best-performing checkpoint per fold according to the same criterion used during training.

3) Mixed-precision inference (bf16) with stable probabilities
Inference runs under bf16 autocast on GPU for efficiency. To ensure numerically stable probabilities, logits are cast back to fp32 before applying softmax to obtain p(tb).
We save per-sample outputs per fold: predictions_fold{k}.csv containing filename, y_true, y_pred, p_tb

4) Confusion matrix contract (single standardized function)
We compute confusion terms using the study label convention and build the confusion matrix in the required layout:
rows = Predicted
cols = True
class order = [Normal, TB]
Each fold saves confusion_fold{k}.png and metrics_fold{k}.csv.

5) Cross-fold aggregation and ROC curves
After all folds are processed, we write: test_inference_metrics.csv (all folds) and print mean ± std for all metrics across folds. We also plot:
per-fold ROC curves
mean ROC curve (interpolated on a common FPR grid)
The ROC plot is saved as: test_roc_curves.png

6) Aggregates all folds into test_inference_metrics.csv and plots ROC curves (only if both classes are present in the test labels).
Important guard
If the test set contains only one class (e.g., only Normal controls), ROC/AUC is undefined; the code prints a warning and skips ROC for those folds.

In [ ]:
# Test inference

OUTPUT_DIR = Path(h["output_dir"])
MODEL_NAME = h["model"]
TEST_DIR = Path(h["test_dir"])

# Encoding used everywhere in your pipelines:
#   normal     -> 0  (plot label: Normal)
#   normal_not -> 1  (plot label: TB)
PLOT_LABELS = ["Normal", "TB"]

# Model summary helper
def print_and_save_model_summary(
    model: torch.nn.Module,
    device: torch.device,
    img_size: int,
    out_txt: Path,
) -> None:
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params

    header = "\n".join(
        [
            "=" * 80,
            f"Model: {MODEL_NAME}",
            f"Device: {device.type}",
            f"Input size (C,H,W): (3, {img_size}, {img_size})",
            f"Total params:      {total_params:,}",
            f"Trainable params:  {trainable_params:,}",
            f"Frozen params:     {frozen_params:,}",
            "=" * 80,
        ]
    )
    print(header)

    table_text = ""
    try:
        from torchsummary import summary as ts_summary  # optional

        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            ts_summary(model, input_size=(3, img_size, img_size), device=("cuda" if device.type == "cuda" else "cpu"))
        table_text = buf.getvalue()
        print(table_text)
    except Exception as e:
        table_text = (
            "\n[torchsummary not available or failed]\n"
            "Falling back to parameter counts only.\n"
            f"Error: {e}\n"
        )
        print(table_text)

    out_txt.parent.mkdir(parents=True, exist_ok=True)
    out_txt.write_text(header + "\n" + table_text)

# Direct test dataset (removes dependency on DataModule internals + allows loader guards)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

class LateralTestDataset(torch.utils.data.Dataset):
    def __init__(self, root: Path, img_size: int) -> None:
        self.root = Path(root)
        self.img_size = int(img_size)

        normal_dir = self.root / "normal"
        tb_dir = self.root / "normal_not"

        normal_paths = sorted([p for p in normal_dir.rglob("*") if p.suffix.lower() in IMG_EXTS])
        tb_paths = sorted([p for p in tb_dir.rglob("*") if p.suffix.lower() in IMG_EXTS])

        self.paths: List[Path] = normal_paths + tb_paths
        self.labels: np.ndarray = np.array([0] * len(normal_paths) + [1] * len(tb_paths), dtype=np.int64)

        self.tf = A.Compose(
            [
                A.Resize(self.img_size, self.img_size),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
                ToTensorV2(),
            ]
        )

        if len(self.paths) == 0:
            raise RuntimeError(f"No images found under: {self.root} (expected subfolders: normal/, normal_not/)")

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        p = self.paths[idx]
        y = int(self.labels[idx])

        img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
        if img is None:
            raise RuntimeError(f"Failed to read image: {p}")

        # handle grayscale (H,W) -> (H,W,3)
        if img.ndim == 2:
            img = np.stack([img, img, img], axis=-1)
        elif img.ndim == 3 and img.shape[2] == 1:
            img = np.repeat(img, 3, axis=2)

        # BGR -> RGB
        if img.ndim == 3 and img.shape[2] == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        x = self.tf(image=img)["image"]  # torch.Tensor (3,H,W)
        return x, y, p.name

def _safe_loader_kwargs(batch_size: int, num_workers: int, pin_memory: bool, device: torch.device) -> Dict:
    cpu_cnt = os.cpu_count() or 1
    nw = int(max(0, min(num_workers, cpu_cnt)))

    use_pin = bool(pin_memory) and (device.type == "cuda")
    kw: Dict = {
        "batch_size": int(batch_size),
        "shuffle": False,
        "num_workers": nw,
        "pin_memory": use_pin,
        "drop_last": False,
    }

    # Only valid when num_workers > 0
    if nw > 0:
        kw["prefetch_factor"] = 2
        kw["persistent_workers"] = True

    return kw

# Test inference across CV folds (each fold model evaluated on the SAME holdout test set)
def inference_test_across_folds(h: Dict) -> None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Output folder
    save_root = OUTPUT_DIR / MODEL_NAME / "test_inference"
    save_root.mkdir(parents=True, exist_ok=True)

    # Build test loader once
    ds_test = LateralTestDataset(TEST_DIR, img_size=int(h["img_size"]))
    dl_test = torch.utils.data.DataLoader(
        ds_test,
        **_safe_loader_kwargs(
            batch_size=int(h["batch_size"]),
            num_workers=int(h["num_workers"]),
            pin_memory=bool(h["pin_memory"]),
            device=device,
        ),
    )

    # AMP context (CPU-safe)
    amp_ctx = torch.cuda.amp.autocast(enabled=(device.type == "cuda"), dtype=torch.float16)

    records: List[Dict] = []
    fpr_list: List[np.ndarray] = []
    tpr_list: List[np.ndarray] = []
    auc_list: List[float] = []

    fold_colors = ["red", "blue", "green", "orange", "brown"]
    axis_fontsize = 12

    # Save model summary only once (optional)
    SAVE_MODEL_SUMMARY_ONCE = True
    summary_written = False

    for fold in range(int(h["n_splits"])):
        ckpt_dir = OUTPUT_DIR / f"fold_{fold}" / MODEL_NAME
        if not ckpt_dir.is_dir():
            print(f"⚠️ Skipping fold {fold}: {ckpt_dir} not found")
            continue

        ckpts = sorted(glob.glob(str(ckpt_dir / f"{MODEL_NAME}_fold{fold}_epoch*_*.ckpt")))
        if not ckpts:
            print(f"⚠️ Skipping fold {fold}: no checkpoints")
            continue

        # pick ckpt with lowest val_loss from filename: ..._epochXX_<val_loss>_<val_mcc>.ckpt
        best_loss = float("inf")
        best_ckpt: Optional[str] = None
        pat = re.compile(rf".*_fold{fold}_epoch\d+_(\d+\.\d+)_\d+\.\d+\.ckpt$")
        for p in ckpts:
            m = pat.match(os.path.basename(p))
            if not m:
                continue
            loss = float(m.group(1))
            if loss < best_loss:
                best_loss = loss
                best_ckpt = p

        if best_ckpt is None:
            print(f"⚠️ No valid ckpt for fold {fold}")
            continue

        print(f"\n🎯 Fold {fold}, ckpt={Path(best_ckpt).name}, val_loss={best_loss:.4f}")

        # Load model (expects TBModel available in scope)
        model = TBModel.load_from_checkpoint(best_ckpt, h=h)  # noqa: F821
        model.to(device).eval()

        fold_dir = save_root / f"fold_{fold}"
        fold_dir.mkdir(parents=True, exist_ok=True)

        if SAVE_MODEL_SUMMARY_ONCE and (not summary_written):
            summary_txt = fold_dir / "model_summary.txt"
            print_and_save_model_summary(model, device, int(h["img_size"]), summary_txt)
            summary_written = True

        # Inference
        y_true: List[int] = []
        y_pred: List[int] = []
        y_prob: List[float] = []

        with torch.inference_mode():
            for imgs, labels, _fnames in dl_test:
                imgs = imgs.to(device, non_blocking=True)
                labels_np = labels.detach().cpu().numpy().astype(int).tolist()

                with amp_ctx:
                    logits = model(imgs)

                logits = logits.float()
                probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
                preds = logits.argmax(dim=1).detach().cpu().numpy()

                y_true.extend(labels_np)
                y_pred.extend(preds.astype(int).tolist())
                y_prob.extend(probs.astype(float).tolist())

        y_true_np = np.asarray(y_true, dtype=int)
        y_pred_np = np.asarray(y_pred, dtype=int)
        y_prob_np = np.asarray(y_prob, dtype=float)

        # Confusion (compute canonical, then transpose to Pred(rows) x True(cols))
        cm_true_pred = confusion_matrix(y_true_np, y_pred_np, labels=[0, 1])   # [[TN,FP],[FN,TP]]
        cm_pred_true = cm_true_pred.T                                         # [[TN,FN],[FP,TP]]

        TN = int(cm_pred_true[0, 0])
        FN = int(cm_pred_true[0, 1])
        FP = int(cm_pred_true[1, 0])
        TP = int(cm_pred_true[1, 1])

        # Save confusion in reference look
        plot_confusion_pred_true(
            cm_pred_true=cm_pred_true,
            classes=PLOT_LABELS,
            save_path=fold_dir / f"confusion_fold{fold}.png",
        )

        # Metrics
        bal_acc = balanced_accuracy_score(y_true_np, y_pred_np)
        sens = recall_score(y_true_np, y_pred_np, pos_label=1)  # TP/(TP+FN)
        spec = TN / (TN + FP) if (TN + FP) else 0.0
        prec = precision_score(y_true_np, y_pred_np, pos_label=1)
        npv = TN / (TN + FN) if (TN + FN) else 0.0
        f1s = f1_score(y_true_np, y_pred_np, pos_label=1)
        mcc = matthews_corrcoef(y_true_np, y_pred_np)
        kappa = cohen_kappa_score(y_true_np, y_pred_np)
        youden = sens + spec - 1

        fpr, tpr, _ = roc_curve(y_true_np, y_prob_np)
        roc_auc = roc_auc_score(y_true_np, y_prob_np)

        records.append(
            {
                "fold": fold,
                "TN": TN,
                "FP": FP,
                "FN": FN,
                "TP": TP,
                "balanced_accuracy": bal_acc,
                "sensitivity": sens,
                "specificity": spec,
                "precision": prec,
                "NPV": npv,
                "F1_score": f1s,
                "MCC": mcc,
                "Cohen_Kappa": kappa,
                "Youden_J": youden,
                "ROC_AUC": roc_auc,
            }
        )

        fpr_list.append(fpr)
        tpr_list.append(tpr)
        auc_list.append(float(roc_auc))

        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    # Save metrics CSV
    df = pd.DataFrame(records)
    out_csv = save_root / "test_inference_metrics.csv"
    df.to_csv(out_csv, index=False)
    print(f"\n✅ Saved metrics to {out_csv}")

    # Cross-fold mean±std
    if not df.empty:
        stats = df.drop(columns=["fold"]).agg(["mean", "std"])
        print("\n📊 Cross-Fold Summary Statistics")
        for metric in stats.columns:
            mu, sd = float(stats.loc["mean", metric]), float(stats.loc["std", metric])
            print(f"  • {metric}: {mu:.4f} ± {sd:.4f}")

    # Plot ROC curves (per-fold + mean)
    if fpr_list:
        common_fpr = np.linspace(0.0, 1.0, 200)
        tprs_interp = [np.interp(common_fpr, fpr_i, tpr_i) for fpr_i, tpr_i in zip(fpr_list, tpr_list)]
        mean_tpr = np.mean(tprs_interp, axis=0)
        mean_auc = float(sk_auc(common_fpr, mean_tpr))

        fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
        for i, (fpr_i, tpr_i, auc_i) in enumerate(zip(fpr_list, tpr_list, auc_list)):
            ax_roc.plot(
                fpr_i,
                tpr_i,
                color=fold_colors[i % len(fold_colors)],
                alpha=0.9,
                lw=2,
                label=f"fold-{i}: AUC={auc_i:.4f}",
            )

        ax_roc.plot(
            common_fpr,
            mean_tpr,
            color="black",
            lw=2,
            linestyle="--",
            label=f"Mean ROC (AUC={mean_auc:.4f})",
        )

        ax_roc.set_xlabel("False Positive Rate", fontsize=axis_fontsize)
        ax_roc.set_ylabel("True Positive Rate", fontsize=axis_fontsize)
        ax_roc.legend(loc="best", fontsize=10)
        ax_roc.set_xlim(0.0, 1.0)
        ax_roc.set_ylim(0.0, 1.02)
        ax_roc.grid(False)

        plt.tight_layout()
        roc_fig_path = save_root / "test_roc_curves.png"
        fig_roc.savefig(str(roc_fig_path), dpi=300, bbox_inches="tight")
        print(f"\n✅ Saved ROC curves plot → {roc_fig_path}")
        plt.show()

    print("\n🎉 Test inference and evaluation complete!")

# Run
inference_test_across_folds(h)

# Statistical Analysis (Best Model → Paired Diagnostic + DCA + Reliability)

**Inputs**
- `h`: experiment hyperparameters (model name, output_dir, test_dir, etc.).
- `CKPT_PATH`: hardcoded path to the single best model checkpoint (selected by lowest validation loss across folds).
- `RAD_CSV`: radiologist predictions metadata CSV (must include filename + three reader prediction columns + MC_status).

**Outputs (saved under `OUT_ROOT/`)**
1) **Diagnostic performance (paired)**:
   - Model vs each radiologist on the same independent test cases.
   - McNemar (overall + TB-positive subset) and paired bootstrap 95% CI for ΔYouden’s J at τ=0.5.
   - Confusion matrices (reference orientation) and ROC with radiologist operating points.
2) **Clinical utility**:
   - Decision Curve Analysis (net benefit curves) for Model, Readers, Treat-All, Treat-None.
3) **Reliability**:
   - Pairwise Cohen’s κ among radiologists and Model-vs-Reader κ.
   - Fleiss κ computed **only** among the 3 radiologists (no model-included Fleiss κ).

Notes:
- Test inference uses a *direct* holdout dataset reader that supports either `test/normal + test/tb` **or**
  `test/normal + test/normal_not` folder naming to prevent silent empty-class loading.
- Confusion matrices are always plotted using `plot_confusion_pred_true(cm_pred_true=[[TN,FN],[FP,TP]])`.


In [ ]:
# Logging
LOG = logging.getLogger("fold_vs_radiologists_lateral")
if not LOG.handlers:
    hnd = logging.StreamHandler()
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    hnd.setFormatter(fmt)
    LOG.addHandler(hnd)
LOG.setLevel(logging.INFO)

# User paths (edit if needed)
CKPT_PATH = Path(
    "/path/to/the/best/model.ckpt"
)

RAD_CSV = Path(
    "/path/to/radiologist/predictions.csv"
)

OUT_ROOT = Path(
    "/path/to/results/folder"
)
# Plot + bootstrap controls
@dataclass(frozen=True)
class PlotCfg:
    cm_figsize: Tuple[int, int] = (6, 6)
    cm_alpha: float = 0.95
    cm_xt_rot: int = 0
    cm_xt_size: int = 14
    cm_yt_size: int = 14
    cm_xlabel_size: int = 14
    cm_ylabel_size: int = 14
    cm_cbar_tick_size: int = 14
    cm_text_size: int = 20
    cm_text_color: str = "black"
    roc_axis_fontsize: int = 14
    roc_legend_fontsize: int = 20
    dis_figsize: Tuple[int, int] = (5, 5)
    dis_axis_size: int = 12
    dis_text_size: int = 18
    upset_figsize: Tuple[int, int] = (10, 6)
    upset_bar_label_size: int = 11
    upset_set_label_size: int = 12
    upset_dot_size: int = 70

PLOT = PlotCfg()

@dataclass(frozen=True)
class BootstrapCfg:
    B_metrics_ci: int = 5000
    B_deltaJ: int = 5000
    seed: int = 1337

BOOT = BootstrapCfg()

# Helpers
def _mcc_from_counts(TN: int, FP: int, FN: int, TP: int) -> float:
    den = float((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN)) ** 0.5
    if den == 0.0:
        return float("nan")
    return float((TP * TN - FP * FN) / den)

def _kappa_from_counts(TN: int, FP: int, FN: int, TP: int) -> float:
    n = TN + FP + FN + TP
    if n == 0:
        return float("nan")

    po = (TP + TN) / n
    pred_pos = TP + FP
    pred_neg = TN + FN
    true_pos = TP + FN
    true_neg = TN + FP
    pe = (pred_pos / n) * (true_pos / n) + (pred_neg / n) * (true_neg / n)

    if (1.0 - pe) <= 0.0:
        return float("nan")
    return float((po - pe) / (1.0 - pe))

def _ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def _basename(x: Any) -> str:
    return Path(str(x)).name

def _is_lateral_lat(fname: str) -> bool:
    """
    Keep lateral filenames robustly.
    Matches stems containing '-LAT' or '_LAT' (case-insensitive), e.g. 'AN2_0011-LAT.jpg'.
    """
    s = _basename(fname)
    stem = s.rsplit(".", 1)[0]
    return re.search(r"(?:^|[_-])lat(?:$|[_-])", stem, flags=re.IGNORECASE) is not None or \
           re.search(r"(?:^|[_-])lat$", stem, flags=re.IGNORECASE) is not None or \
           re.search(r"-lat$", stem, flags=re.IGNORECASE) is not None

def _to_bin_pred(x: Any) -> float:
    """
    Radiologist predictions are typically 'Cases'/'Controls'. Map:
      - Cases / case / TB / positive / 1 / true  -> 1
      - Controls / control / Normal / negative / 0 / false -> 0
    Returns np.nan if value cannot be interpreted.
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    s = str(x).strip().lower()
    if s in {"cases", "case", "tb", "tuberculosis", "pos", "positive", "1", "true", "yes"}:
        return 1.0
    if s in {"controls", "control", "normal", "neg", "negative", "0", "false", "no"}:
        return 0.0
    return np.nan

def _safe_div(num: float, den: float) -> float:
    return float(num) / float(den) if den else 0.0

# Correct TN/FP/FN/TP extraction (positive class = 1 = TB)
def _counts_from_y(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[int, int, int, int]:
    """
    Returns (TN, FP, FN, TP) using standard definitions.
    """
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    TN = int(((y_true == 0) & (y_pred == 0)).sum())
    FP = int(((y_true == 0) & (y_pred == 1)).sum())
    FN = int(((y_true == 1) & (y_pred == 0)).sum())
    TP = int(((y_true == 1) & (y_pred == 1)).sum())
    return TN, FP, FN, TP

def _confusion_predrows_truecols(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """
    layout (Predicted rows, True columns):
        [[TN, FN],
         [FP, TP]]
    """
    TN, FP, FN, TP = _counts_from_y(y_true, y_pred)
    return np.array([[TN, FN],
                     [FP, TP]], dtype=int)
    
def _has_both_classes(y: np.ndarray) -> bool:
    y = np.asarray(y, dtype=int)
    u = np.unique(y)
    return u.size >= 2
    
def _metrics_from_outputs(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    TN, FP, FN, TP = _counts_from_y(y_true, y_pred)

    sens = _safe_div(TP, TP + FN)
    spec = _safe_div(TN, TN + FP)
    bal_acc = 0.5 * (sens + spec)
    prec = _safe_div(TP, TP + FP)
    npv = _safe_div(TN, TN + FN)
    f1s = _safe_div(2.0 * TP, 2.0 * TP + FP + FN)
    mcc = _mcc_from_counts(TN, FP, FN, TP)
    kappa = _kappa_from_counts(TN, FP, FN, TP)
    youden = sens + spec - 1.0

    out: Dict[str, float] = dict(
        TP=float(TP), FP=float(FP), FN=float(FN), TN=float(TN),
        balanced_accuracy=float(bal_acc),
        sensitivity=float(sens),
        specificity=float(spec),
        precision=float(prec),
        NPV=float(npv),
        F1_score=float(f1s),
        MCC=float(mcc),
        Cohen_Kappa=float(kappa),
        Youden_J=float(youden),
    )
    if (y_prob is not None) and _has_both_classes(y_true):
        try:
            out["ROC_AUC"] = float(roc_auc_score(y_true, y_prob))
        except Exception:
            out["ROC_AUC"] = float("nan")
    else:
        out["ROC_AUC"] = float("nan")
    return out

def _youden_j_from_preds(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    TN, FP, FN, TP = _counts_from_y(y_true, y_pred)
    sens = _safe_div(TP, TP + FN)
    spec = _safe_div(TN, TN + FP)
    return float(sens + spec - 1.0)

# Confusion plot: Predicted rows (y), True columns (x)
def plot_confusion_pred_true(
    cm: np.ndarray,
    class_names: List[str],
    save_path: Path,
    *,
    figsize: Tuple[int, int] = (8, 6),
    cmap: str = "Purples",
    alpha: float = 0.85,
    xt_label_rotation: int = 0,
    xt_label_size: int = 22,
    yt_label_size: int = 22,
    xlabel_font_size: int = 26,
    ylabel_font_size: int = 26,
    cbar_tick_size: int = 22,
    text_color: str = "black",
    text_font_size: int = 38,
) -> None:
    """
    cm layout:
        [[TN, FN],
         [FP, TP]]
    """
    cm = np.asarray(cm, dtype=int)
    if cm.shape != (2, 2):
        raise ValueError(f"Expected 2x2 cm, got {cm.shape}")

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.matshow(cm, cmap=plt.get_cmap(cmap), alpha=alpha)

    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.ax.tick_params(labelsize=cbar_tick_size)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(class_names, rotation=xt_label_rotation, fontsize=xt_label_size)
    ax.set_yticklabels(class_names, fontsize=yt_label_size)

    # Force x ticks bottom like your reference
    ax.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)

    ax.set_xlabel("True Label", fontsize=xlabel_font_size)
    ax.set_ylabel("Predicted Label", fontsize=ylabel_font_size)

    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(int(v)), ha="center", va="center",
                fontsize=text_font_size, color=text_color)

    plt.tight_layout()
    _ensure_dir(save_path.parent)
    fig.savefig(save_path, dpi=400, bbox_inches="tight")
    plt.close(fig)

# EXACT inference (kept semantically identical)
@torch.no_grad()
def infer_exact(
    h: Dict,
    ckpt_path: Path,
) -> pd.DataFrame:
    if not ckpt_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    from contextlib import nullcontext
    if device.type == "cuda":
        # PyTorch >= 2.0 preferred API
        try:
            amp_ctx = torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16)
        except AttributeError:
            # fallback for older torch
            amp_ctx = torch.cuda.amp.autocast(dtype=torch.bfloat16)
    else:
        amp_ctx = nullcontext()

    dm_test = DataModule(h, fold_idx=None)
    dm_test.setup()
    test_loader = dm_test.test_dataloader()

    model = TBModel.load_from_checkpoint(str(ckpt_path), h=h)
    model.to(device).eval()

    y_true: List[int] = []
    y_pred: List[int] = []
    y_prob: List[float] = []
    fnames: List[str] = []

    with torch.inference_mode():
        for batch in test_loader:
            if not isinstance(batch, (list, tuple)):
                raise ValueError("Unexpected batch type from DataLoader")

            if len(batch) == 3:
                imgs, labels, _filenames = batch
            elif len(batch) == 2:
                imgs, labels = batch
                _filenames = None
            else:
                raise ValueError(f"Unexpected batch structure with length {len(batch)}")

            imgs = imgs.to(device, non_blocking=(device.type == "cuda"))

            with amp_ctx:
                logits = model(imgs)

            logits = logits.float()
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            preds = logits.argmax(dim=1).detach().cpu().numpy()

            if torch.is_tensor(labels):
                labels_np = labels.detach().cpu().numpy().astype(int)
            else:
                labels_np = np.asarray(labels, dtype=int)

            y_true.extend(labels_np.tolist())
            y_pred.extend(preds.tolist())
            y_prob.extend(probs.tolist())

            if _filenames is not None:
                fnames.extend([_basename(x) for x in _filenames])

    y_true_arr = np.asarray(y_true, dtype=int)
    y_pred_arr = np.asarray(y_pred, dtype=int)
    y_prob_arr = np.asarray(y_prob, dtype=float)

    if fnames and (len(fnames) != len(y_true_arr)):
        raise RuntimeError(f"Filename count mismatch: {len(fnames)} vs labels {len(y_true_arr)}")

    out = pd.DataFrame({
        "original": fnames if fnames else [f"idx_{i:05d}" for i in range(len(y_true_arr))],
        "y_true": y_true_arr,
        "model_pred": y_pred_arr,
        "model_prob": y_prob_arr,
    })

    # Optional sanity: ensure these are laterals (won't crash if DataLoader yields different naming)
    if "original" in out.columns:
        kept = out["original"].map(_is_lateral_lat)
        if kept.mean() < 0.95:
            LOG.warning(f"[QC] Only {100.0*kept.mean():.1f}% of inference filenames look like LAT by naming.")

    return out

# ROC plot - legend contains AUC/J
def plot_roc_model_plus_rads(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    rad_points: Dict[str, Tuple[float, float, float]],  # name -> (FPR, TPR, J)
    model_J: float,
    model_auc: float,
    model_point: Tuple[float, float],
    save_path: Path,
) -> None:
    fpr, tpr, _ = roc_curve(y_true, y_prob)

    pt_color = {
        "Model": "green",
        "Rad-1": "red",
        "Rad-2": "blue",
        "Rad-3": "saddlebrown",
    }

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(fpr, tpr, color="red", lw=2)

    ax.scatter([model_point[0]], [model_point[1]], s=90, marker="o",
               color=pt_color["Model"], edgecolors="black")

    marker_map = {"Rad-1": "^", "Rad-2": "s", "Rad-3": "D"}
    for name, (xfpr, ytpr, _) in rad_points.items():
        ax.scatter([xfpr], [ytpr], s=90, marker=marker_map.get(name, "^"),
                   color=pt_color.get(name, "black"), edgecolors="black")

    ax.set_xlabel("False Positive Rate", fontsize=PLOT.roc_axis_fontsize)
    ax.set_ylabel("True Positive Rate", fontsize=PLOT.roc_axis_fontsize)
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.02)
    ax.grid(False)

    handles: List[Line2D] = []
    handles.append(Line2D(
        [0], [0], color="red", lw=2,
        marker="o", markerfacecolor=pt_color["Model"], markeredgecolor="black",
        label=f"Model (AUC={model_auc:.4f}, J={model_J:.4f})"
    ))
    for rname in ["Rad-1", "Rad-2", "Rad-3"]:
        if rname in rad_points:
            jv = float(rad_points[rname][2])
            handles.append(Line2D(
                [0], [0],
                color="none", lw=0,
                marker=marker_map.get(rname, "^"),
                markerfacecolor=pt_color[rname],
                markeredgecolor="black",
                label=f"{rname} (J={jv:.4f})"
            ))
    ax.legend(handles=handles, loc="best", fontsize=PLOT.roc_legend_fontsize)

    plt.tight_layout()
    _ensure_dir(save_path.parent)
    fig.savefig(save_path, dpi=400, bbox_inches="tight")
    plt.close(fig)

# McNemar exact + Holm + paired bootstrap ΔJ
def _mcnemar_exact_pvalue(b: int, c: int) -> float:
    n = b + c
    if n == 0:
        return 1.0
    try:
        from scipy.stats import binomtest
        return float(binomtest(k=min(b, c), n=n, p=0.5, alternative="two-sided").pvalue)
    except Exception:
        try:
            from scipy.stats import chi2
            chi2_stat = (abs(b - c) - 1.0) ** 2 / float(n)
            return float(1.0 - chi2.cdf(chi2_stat, df=1))
        except Exception:
            return float("nan")

def _holm_bonferroni(pvals: Dict[str, float]) -> Dict[str, float]:
    """Holm–Bonferroni step-down adjustment.

    NaN (or non-finite) p-values are preserved as NaN and excluded from ordering/adjustment.
    """
    items_all = list(pvals.items())
    finite_items = [(k, float(p)) for (k, p) in items_all if (p is not None) and np.isfinite(float(p))]
    finite_items.sort(key=lambda kv: kv[1])

    m = len(finite_items)
    adj: Dict[str, float] = {}

    # If nothing finite, preserve NaNs
    if m == 0:
        for k, _ in items_all:
            adj[k] = float("nan")
        return adj

    prev = 0.0
    for i, (k, p) in enumerate(finite_items, start=1):
        a = min(1.0, (m - i + 1) * p)
        a = max(a, prev)  # monotone
        adj[k] = float(a)
        prev = float(a)

    # Put back NaNs for excluded tests
    for k, p in items_all:
        if k not in adj:
            adj[k] = float("nan")
    return adj

def paired_bootstrap_delta_J(
    y_true: np.ndarray,
    y_pred_model: np.ndarray,
    y_pred_reader: np.ndarray,
    B: int,
    seed: int,
) -> Tuple[float, float, float]:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    if not (n == len(y_pred_model) == len(y_pred_reader)):
        raise ValueError("Length mismatch in paired_bootstrap_delta_J")

    delta0 = _youden_j_from_preds(y_true, y_pred_model) - _youden_j_from_preds(y_true, y_pred_reader)

    deltas = np.empty(B, dtype=float)
    idx = np.arange(n)
    for b in range(B):
        samp = rng.choice(idx, size=n, replace=True)
        deltas[b] = _youden_j_from_preds(y_true[samp], y_pred_model[samp]) - _youden_j_from_preds(y_true[samp], y_pred_reader[samp])

    ci_low, ci_high = np.quantile(deltas, [0.025, 0.975]).tolist()
    return float(delta0), float(ci_low), float(ci_high)

# Bootstrap CI for metrics (Model + each reader)
def bootstrap_metrics_ci(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: Optional[np.ndarray],
    B: int,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    if y_prob is not None:
        y_prob = np.asarray(y_prob, dtype=float)

    point = _metrics_from_outputs(y_true, y_pred, y_prob)
    keys = list(point.keys())
    boot = {k: [] for k in keys}

    idx = np.arange(len(y_true))
    for _ in range(B):
        samp = rng.choice(idx, size=len(idx), replace=True)
        yt = y_true[samp]
        yp = y_pred[samp]
        yp_prob = (y_prob[samp] if y_prob is not None else None)
        m = _metrics_from_outputs(yt, yp, yp_prob)
        for k in keys:
            boot[k].append(m[k])

    rows = []
    for k in keys:
        arr = np.asarray(boot[k], dtype=float)
        finite = arr[np.isfinite(arr)]
        if finite.size == 0:
            lo = hi = float("nan")
        else:
            lo, hi = np.quantile(finite, [0.025, 0.975]).tolist()
        rows.append({"metric": k, "point": float(point[k]), "ci_low": float(lo), "ci_high": float(hi)})
        # arr = np.asarray(boot[k], dtype=float)
        # lo, hi = np.nanquantile(arr, [0.025, 0.975]).tolist()
        # rows.append({"metric": k, "point": float(point[k]), "ci_low": float(lo), "ci_high": float(hi)})

    return pd.DataFrame(rows)

# TB+ discordance square
DISCORDANCE_CMAPS = {"All": "Greens", "Confirmed": "Blues", "Unconfirmed": "YlOrBr"}

def plot_tbpos_discordance_square(
    m_hit: np.ndarray,
    r_hit: np.ndarray,
    save_path: Path,
    *,
    reader_name: str,
    cohort: str,
    cmap: str = "Greys",
) -> Tuple[int, int]:
    m_hit = np.asarray(m_hit, dtype=bool)
    r_hit = np.asarray(r_hit, dtype=bool)
    if len(m_hit) != len(r_hit):
        raise ValueError("m_hit/r_hit length mismatch")

    both_miss = int((~m_hit & ~r_hit).sum())
    reader_only = int((~m_hit &  r_hit).sum())  # c
    model_only = int(( m_hit & ~r_hit).sum())  # b
    both_hit = int(( m_hit &  r_hit).sum())

    mat = np.array([[both_miss, reader_only],
                    [model_only,  both_hit]], dtype=int)

    fig, ax = plt.subplots(figsize=PLOT.dis_figsize)
    im = ax.matshow(mat, cmap=plt.get_cmap(cmap), alpha=0.9)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.ax.tick_params(labelsize=PLOT.dis_axis_size)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Normal", "TB"], fontsize=PLOT.dis_axis_size)
    ax.set_yticklabels(["Normal", "TB"], fontsize=PLOT.dis_axis_size)
    ax.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)
    ax.set_xlabel(f"{reader_name}", fontsize=PLOT.dis_axis_size)
    ax.set_ylabel("Model", fontsize=PLOT.dis_axis_size)

    for (i, j), v in np.ndenumerate(mat):
        ax.text(j, i, str(int(v)), ha="center", va="center", fontsize=PLOT.dis_text_size)

    plt.tight_layout()
    _ensure_dir(save_path.parent)
    fig.savefig(save_path, dpi=400, bbox_inches="tight")
    plt.close(fig)

    return model_only, reader_only

# UpSet-like intersections
def _pattern_counts(bits: np.ndarray) -> pd.DataFrame:
    bits = np.asarray(bits, dtype=int)
    K = bits.shape[1]
    patterns = ["".join(map(str, row.tolist())) for row in bits]
    vc = pd.Series(patterns).value_counts()
    out = vc.reset_index()
    out.columns = ["pattern", "count"]
    for k in range(K):
        out[f"b{k}"] = out["pattern"].str[k].astype(int)
    return out.sort_values("count", ascending=False).reset_index(drop=True)

def plot_upset_like(
    bits: np.ndarray,
    set_names: List[str],
    set_colors: List[str],
    save_path: Path,
    *,
    caption: str,
) -> pd.DataFrame:
    if not (bits.shape[1] == len(set_names) == len(set_colors)):
        raise ValueError("bits columns must match set_names/set_colors length")

    tbl = _pattern_counts(bits)
    patterns = tbl["pattern"].tolist()
    counts = tbl["count"].to_numpy(dtype=int)

    K = len(set_names)
    M = len(patterns)
    x = np.arange(M)

    fig = plt.figure(figsize=PLOT.upset_figsize, constrained_layout=True)
    gs = fig.add_gridspec(2, 1, height_ratios=[2.0, 1.6]) 
    ax_bar = fig.add_subplot(gs[0, 0])
    ax_mat = fig.add_subplot(gs[1, 0], sharex=ax_bar)

    ax_bar.bar(x, counts)
    ax_bar.set_ylabel("Count", fontsize=PLOT.upset_bar_label_size)
    ax_bar.tick_params(axis="x", labelbottom=False)
    ax_bar.tick_params(axis="y", labelsize=PLOT.upset_bar_label_size)

    ax_mat.set_yticks(np.arange(K))
    ax_mat.set_yticklabels(list(reversed(set_names)), fontsize=PLOT.upset_set_label_size)
    ax_mat.set_xlabel(caption, fontsize=11)
    ax_mat.tick_params(axis="x", labelsize=10)

    mem = np.stack([tbl[f"b{k}"].to_numpy(dtype=int) for k in range(K)], axis=1)  # (M,K)
    mem = mem[:, ::-1]

    for j in range(M):
        ys_on = np.where(mem[j] == 1)[0]
        for yi in range(K):
            ax_mat.scatter(j, yi, s=PLOT.upset_dot_size * 0.35, color="lightgray", edgecolors="none")
        for yi in ys_on:
            color = set_colors[::-1][yi]
            ax_mat.scatter(j, yi, s=PLOT.upset_dot_size, color=color, edgecolors="black", linewidths=0.6)
        if len(ys_on) >= 2:
            ax_mat.plot([j, j], [ys_on.min(), ys_on.max()], color="black", lw=1.0)

    ax_mat.set_xlim(-0.6, M - 0.4)
    ax_mat.set_ylim(-0.6, K - 0.4)

    plt.tight_layout()
    _ensure_dir(save_path.parent)
    fig.savefig(save_path, dpi=400, bbox_inches="tight")
    plt.close(fig)
    return tbl

# Reader agreement: pairwise κ + Fleiss κ (3 raters, binary)
def cohen_kappa_binary(y_a: np.ndarray, y_b: np.ndarray) -> float:
    """Cohen's κ between two binary raters (0/1), computed without external deps.

    NaNs should be handled by caller (e.g., by masking before passing).
    """
    a = np.asarray(y_a, dtype=int)
    b = np.asarray(y_b, dtype=int)
    if len(a) != len(b):
        raise ValueError("Length mismatch in cohen_kappa_binary")
    # Treat a as reference and b as prediction; κ is symmetric for binary labels.
    TN = int(((a == 0) & (b == 0)).sum())
    FP = int(((a == 0) & (b == 1)).sum())
    FN = int(((a == 1) & (b == 0)).sum())
    TP = int(((a == 1) & (b == 1)).sum())
    return float(_kappa_from_counts(TN, FP, FN, TP))

def fleiss_kappa_binary(ratings_NxK: np.ndarray) -> float:
    """Fleiss' κ for binary ratings (0/1) for K>=2 raters.

    Args:
        ratings_NxK: array of shape (N, K) with integer labels in {0,1}.
            Missing values must be handled by the caller (mask rows first).

    Returns:
        Fleiss' κ (float). Returns NaN if undefined (e.g., Pe==1) or if N==0.
    """
    R = np.asarray(ratings_NxK)
    if not (R.ndim == 2 and R.shape[1] >= 2):
        raise ValueError("ratings must be shape (N,K) with K>=2")

    if R.size == 0:
        return float("nan")
    if np.isnan(R).any():
        raise ValueError("ratings contains NaNs; mask missing rows before calling fleiss_kappa_binary")

    R = R.astype(int)
    if not np.isin(R, [0, 1]).all():
        raise ValueError("ratings must be binary (0/1)")

    N = int(R.shape[0])
    n = int(R.shape[1])

    # Per-item category counts
    n0 = (R == 0).sum(axis=1).astype(float)
    n1 = (R == 1).sum(axis=1).astype(float)

    # Per-item agreement
    denom = float(n * (n - 1))
    P_i = (n0 * (n0 - 1.0) + n1 * (n1 - 1.0)) / denom
    Pbar = float(P_i.mean())

    # Marginal category proportions
    p0 = float(n0.sum()) / float(N * n)
    p1 = float(n1.sum()) / float(N * n)
    Pe = p0 * p0 + p1 * p1

    if (1.0 - Pe) <= 0.0:
        return float("nan")
    return float((Pbar - Pe) / (1.0 - Pe))

def reader_agreement_summary(rad1: np.ndarray, rad2: np.ndarray, rad3: np.ndarray) -> Dict[str, Any]:
    r1 = np.asarray(rad1, dtype=float)
    r2 = np.asarray(rad2, dtype=float)
    r3 = np.asarray(rad3, dtype=float)
    keep = (~np.isnan(r1)) & (~np.isnan(r2)) & (~np.isnan(r3))

    r1i = r1[keep].astype(int)
    r2i = r2[keep].astype(int)
    r3i = r3[keep].astype(int)

    out: Dict[str, Any] = {}
    out["N_all3"] = int(len(r1i))
    if len(r1i) == 0:
        out.update({
            "kappa_R1_R2": np.nan, "kappa_R1_R3": np.nan, "kappa_R2_R3": np.nan,
            "fleiss_kappa": np.nan,
            "count_3of3_TB": 0, "count_3of3_Normal": 0, "count_2of3_TB": 0, "count_1of3_TB": 0,
        })
        return out

    out["kappa_R1_R2"] = float(cohen_kappa_binary(r1i, r2i))
    out["kappa_R1_R3"] = float(cohen_kappa_binary(r1i, r3i))
    out["kappa_R2_R3"] = float(cohen_kappa_binary(r2i, r3i))
    out["fleiss_kappa"] = float(fleiss_kappa_binary(np.stack([r1i, r2i, r3i], axis=1)))

    s = r1i + r2i + r3i
    out["count_3of3_TB"] = int((s == 3).sum())
    out["count_3of3_Normal"] = int((s == 0).sum())
    out["count_2of3_TB"] = int((s == 2).sum())
    out["count_1of3_TB"] = int((s == 1).sum())
    return out

# Error analysis: FN/FP + rescues (requires all 3 readers for intersections)
def error_analysis(
    y_true: np.ndarray,
    model_pred: np.ndarray,
    rad1: np.ndarray,
    rad2: np.ndarray,
    rad3: np.ndarray,
) -> pd.DataFrame:
    y_true = np.asarray(y_true, dtype=int)
    m = np.asarray(model_pred, dtype=int)
    r1f = np.asarray(rad1, dtype=float)
    r2f = np.asarray(rad2, dtype=float)
    r3f = np.asarray(rad3, dtype=float)

    rows = []

    def _fn_fp_for(pred_float: np.ndarray, name: str) -> None:
        keep = ~np.isnan(pred_float)
        yt = y_true[keep]
        yp = pred_float[keep].astype(int)
        fn_tb = int(((yt == 1) & (yp == 0)).sum())
        fp_ctl = int(((yt == 0) & (yp == 1)).sum())
        rows.append({"method": name, "FN_TB": fn_tb, "FP_controls": fp_ctl})

    _fn_fp_for(m.astype(float), "Model")
    _fn_fp_for(r1f, "Rad-1")
    _fn_fp_for(r2f, "Rad-2")
    _fn_fp_for(r3f, "Rad-3")

    keep3 = (~np.isnan(r1f)) & (~np.isnan(r2f)) & (~np.isnan(r3f))
    yt3 = y_true[keep3]
    m3 = m[keep3]
    r1 = r1f[keep3].astype(int)
    r2 = r2f[keep3].astype(int)
    r3 = r3f[keep3].astype(int)

    tbpos = (yt3 == 1)

    model_miss_tb = int(((tbpos) & (m3 == 0)).sum())
    r1_miss_tb = int(((tbpos) & (r1 == 0)).sum())
    r2_miss_tb = int(((tbpos) & (r2 == 0)).sum())
    r3_miss_tb = int(((tbpos) & (r3 == 0)).sum())
    all3_miss_tb = int(((tbpos) & (r1 == 0) & (r2 == 0) & (r3 == 0)).sum())

    model_rescue_r1 = int(((tbpos) & (r1 == 0) & (m3 == 1)).sum())
    model_rescue_r2 = int(((tbpos) & (r2 == 0) & (m3 == 1)).sum())
    model_rescue_r3 = int(((tbpos) & (r3 == 0) & (m3 == 1)).sum())
    any_reader_rescue_model = int(((tbpos) & (m3 == 0) & ((r1 == 1) | (r2 == 1) | (r3 == 1))).sum())

    ctl = (yt3 == 0)
    fp_model = int(((ctl) & (m3 == 1)).sum())
    fp_r1 = int(((ctl) & (r1 == 1)).sum())
    fp_r2 = int(((ctl) & (r2 == 1)).sum())
    fp_r3 = int(((ctl) & (r3 == 1)).sum())

    summary = pd.DataFrame([{
        "N_all3_present": int(len(yt3)),
        "N_TBpos_all3_present": int(tbpos.sum()),
        "TB_missed_by_model": model_miss_tb,
        "TB_missed_by_rad1": r1_miss_tb,
        "TB_missed_by_rad2": r2_miss_tb,
        "TB_missed_by_rad3": r3_miss_tb,
        "TB_missed_by_all3_readers": all3_miss_tb,
        "model_rescues_rad1_FNs": model_rescue_r1,
        "model_rescues_rad2_FNs": model_rescue_r2,
        "model_rescues_rad3_FNs": model_rescue_r3,
        "any_reader_rescues_model_FNs": any_reader_rescue_model,
        "FP_controls_model": fp_model,
        "FP_controls_rad1": fp_r1,
        "FP_controls_rad2": fp_r2,
        "FP_controls_rad3": fp_r3,
    }])

    return pd.concat([pd.DataFrame(rows), summary], axis=0, ignore_index=True)

def plot_fp_controls_bar(err_df: pd.DataFrame, save_path: Path, caption: str) -> None:
    per = err_df.loc[err_df["method"].notna()].copy()
    per = per[per["method"].isin(["Model", "Rad-1", "Rad-2", "Rad-3"])]

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(per["method"], per["FP_controls"].astype(int).to_numpy())
    ax.set_ylabel("False Positives on Controls")
    ax.set_xlabel(caption)
    plt.tight_layout()
    _ensure_dir(save_path.parent)
    fig.savefig(save_path, dpi=400, bbox_inches="tight")
    plt.close(fig)

# Cohorts
def build_cohort(df: pd.DataFrame, cohort: str) -> pd.DataFrame:
    c = cohort.strip().lower()
    if c == "all":
        return df.copy()
    if c in ("confirmed", "unconfirmed"):
        status = "Confirmed" if c == "confirmed" else "Unconfirmed"
        t = df["Type"].astype(str).str.strip().str.lower()
        is_ctrl = t.str.startswith("control")
        is_case = t.str.startswith("case")
        mc = df["MC_status"].astype(str).str.strip().str.lower()
        mask = is_ctrl | (is_case & (mc == status.lower()))
        # mask = (df["Type"].str.lower() == "controls") | (
        #     (df["Type"].str.lower() == "cases") & (df["MC_status"].fillna("") == status)
        # )
        return df.loc[mask].copy()
    raise ValueError(f"Unknown cohort: {cohort}")

# Agreement metrics
def _kappa_between(a: np.ndarray, b: np.ndarray) -> Tuple[float, int]:
    """Cohen's κ between two binary raters with automatic NaN masking.

    Returns (kappa, N_used).
    """
    af = np.asarray(a, dtype=float)
    bf = np.asarray(b, dtype=float)
    keep = np.isfinite(af) & np.isfinite(bf)
    if int(keep.sum()) == 0:
        return float("nan"), 0
    ai = af[keep].astype(int)
    bi = bf[keep].astype(int)
    return float(cohen_kappa_binary(ai, bi)), int(len(ai))

def agreement_metrics_exp1(
    *,
    y_true: np.ndarray,
    model_pred: np.ndarray,
    rad1: np.ndarray,
    rad2: np.ndarray,
    rad3: np.ndarray,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Compute agreement metrics for Exp-1.

    Computes:
      - Pairwise Cohen's κ between all pairs in {Truth, Model, Rad-1, Rad-2, Rad-3}
        with automatic pairwise NaN masking.
      - Fleiss' κ among the three radiologists (Rad-1/Rad-2/Rad-3) using rows where all
        three radiologist ratings are present.
      - Fleiss' κ among {Model, Rad-1, Rad-2, Rad-3} using rows where all three
        radiologist ratings are present (model predictions are assumed always present).

    Returns:
      - long_df: tidy table of metrics with columns: pair, kappa, N
      - kappa_mat_df: symmetric κ matrix over {Truth, Model, Rad-1, Rad-2, Rad-3}
        using pairwise Cohen's κ
    """
    yt = np.asarray(y_true, dtype=int)
    mp = np.asarray(model_pred, dtype=int)

    r1f = np.asarray(rad1, dtype=float)
    r2f = np.asarray(rad2, dtype=float)
    r3f = np.asarray(rad3, dtype=float)

    # Pairwise kappas (Truth included as a "rater" for completeness)
    raters: Dict[str, np.ndarray] = {
        "Truth": yt.astype(float),
        "Model": mp.astype(float),
        "Rad-1": r1f,
        "Rad-2": r2f,
        "Rad-3": r3f,
    }

    long_rows: List[Dict[str, Any]] = []
    names = list(raters.keys())
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = raters[names[i]], raters[names[j]]
            kap, n_used = _kappa_between(a, b)
            long_rows.append({"pair": f"{names[i]} vs {names[j]}", "kappa": float(kap), "N": int(n_used)})

    # Fleiss κ among 3 radiologists (requires all 3 present)
    keep3 = np.isfinite(r1f) & np.isfinite(r2f) & np.isfinite(r3f)
    if int(keep3.sum()) == 0:
        fleiss_rads = float("nan")
        n_fleiss_rads = 0
        fleiss_model_rads = float("nan")
        n_fleiss_model_rads = 0
    else:
        r3 = np.stack(
            [r1f[keep3].astype(int), r2f[keep3].astype(int), r3f[keep3].astype(int)],
            axis=1,
        )
        fleiss_rads = float(fleiss_kappa_binary(r3))
        n_fleiss_rads = int(keep3.sum())

        # Fleiss κ among Model + 3 radiologists (same rows as keep3, model assumed finite)
        mpf = mp.astype(float)
        keep4 = keep3 & np.isfinite(mpf)
        r4 = np.stack(
            [mpf[keep4].astype(int), r1f[keep4].astype(int), r2f[keep4].astype(int), r3f[keep4].astype(int)],
            axis=1,
        )
        fleiss_model_rads = float(fleiss_kappa_binary(r4))
        n_fleiss_model_rads = int(keep4.sum())

    long_rows.append({"pair": "Fleiss κ (Rad-1/Rad-2/Rad-3)", "kappa": float(fleiss_rads), "N": int(n_fleiss_rads)})
    long_rows.append({"pair": "Fleiss κ (Model/Rad-1/Rad-2/Rad-3)", "kappa": float(fleiss_model_rads), "N": int(n_fleiss_model_rads)})

    long_df = pd.DataFrame(long_rows).sort_values(["pair"]).reset_index(drop=True)

    # κ matrix (pairwise Cohen's κ only)
    mat = np.full((len(names), len(names)), np.nan, dtype=float)
    np.fill_diagonal(mat, 1.0)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            kap, _ = _kappa_between(raters[names[i]], raters[names[j]])
            mat[i, j] = mat[j, i] = float(kap)

    kappa_mat_df = pd.DataFrame(mat, index=names, columns=names)
    return long_df, kappa_mat_df

def plot_kappa_heatmap(
    kappa_mat: pd.DataFrame,
    save_path: Path,
    *,
    title: str,
    vmin: float = -0.2,
    vmax: float = 1.0,
    cmap: str = "viridis",   
) -> None:
    """Simple κ heatmap without seaborn, with selectable colormap."""
    vals = kappa_mat.to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(8, 7))

    # Pass the colormap here
    im = ax.imshow(vals, vmin=vmin, vmax=vmax, cmap=cmap)

    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.ax.tick_params(labelsize=12)
    ax.set_xticks(range(kappa_mat.shape[1]))
    ax.set_yticks(range(kappa_mat.shape[0]))
    ax.set_xticklabels(kappa_mat.columns.tolist(), rotation=45, ha="right", fontsize=12)
    ax.set_yticklabels(kappa_mat.index.tolist(), fontsize=12)
    ax.set_title(title, fontsize=14)

    for (i, j), v in np.ndenumerate(vals):
        if np.isfinite(v):
            ax.text(j, i, f"{v:.4f}", ha="center", va="center", fontsize=10)

    plt.tight_layout()
    _ensure_dir(save_path.parent)
    fig.savefig(save_path, dpi=400, bbox_inches="tight")
    plt.close(fig)

# Decision curve analysis (DCA)

def _net_benefit_from_counts(tp: int, fp: int, n: int, pt: float) -> float:
    """Net benefit at threshold pt given TP/FP counts on n cases.

    NB(pt) = TP/n - FP/n * (pt/(1-pt))
    """
    if n <= 0 or not (0.0 < pt < 1.0):
        return float("nan")
    w = float(pt) / float(1.0 - pt)
    return float(tp) / float(n) - (float(fp) / float(n)) * w

def decision_curve_from_prob(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    thresholds: np.ndarray,
) -> np.ndarray:
    yt = np.asarray(y_true, dtype=int)
    yp = np.asarray(y_prob, dtype=float)
    out = np.empty(len(thresholds), dtype=float)
    n = int(len(yt))
    for i, pt in enumerate(thresholds):
        pred = (yp >= float(pt)).astype(int)
        tn, fp, fn, tp = _counts_from_y(yt, pred)
        out[i] = _net_benefit_from_counts(int(tp), int(fp), n, float(pt))
    return out

def decision_curve_from_pred(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    thresholds: np.ndarray,
) -> np.ndarray:
    yt = np.asarray(y_true, dtype=int)
    ypf = np.asarray(y_pred, dtype=float)
    keep = np.isfinite(ypf)
    if int(keep.sum()) == 0:
        return np.full(len(thresholds), np.nan, dtype=float)

    yt = yt[keep]
    yp = ypf[keep].astype(int)
    n = int(len(yt))
    tn, fp, fn, tp = _counts_from_y(yt, yp)

    out = np.empty(len(thresholds), dtype=float)
    for i, pt in enumerate(thresholds):
        out[i] = _net_benefit_from_counts(int(tp), int(fp), n, float(pt))
    return out

def bootstrap_delta_net_benefit_ci(
    *,
    y_true: np.ndarray,
    y_prob_model: np.ndarray,
    y_pred_reader: np.ndarray,
    pt_values: np.ndarray,
    B: int,
    seed: int,
) -> pd.DataFrame:
    """Paired bootstrap CI for ΔNB(pt)=NB_model(pt)-NB_reader(pt) for each pt in pt_values."""
    rng = np.random.default_rng(seed)
    yt = np.asarray(y_true, dtype=int)
    pm = np.asarray(y_prob_model, dtype=float)
    prf = np.asarray(y_pred_reader, dtype=float)
    keep = np.isfinite(prf) & np.isfinite(pm)
    yt = yt[keep]
    pm = pm[keep]
    pr = prf[keep].astype(int)

    n = int(len(yt))
    if n == 0:
        return pd.DataFrame({"pt": pt_values, "delta_nb": np.nan, "ci_low": np.nan, "ci_high": np.nan})

    idx = np.arange(n)
    delta_point = decision_curve_from_prob(yt, pm, pt_values) - decision_curve_from_pred(yt, pr, pt_values)

    boot = np.empty((B, len(pt_values)), dtype=float)
    for b in range(B):
        samp = rng.choice(idx, size=n, replace=True)
        yt_b = yt[samp]
        pm_b = pm[samp]
        pr_b = pr[samp]
        boot[b, :] = decision_curve_from_prob(yt_b, pm_b, pt_values) - decision_curve_from_pred(yt_b, pr_b, pt_values)

    ci_low = np.quantile(boot, 0.025, axis=0)
    ci_high = np.quantile(boot, 0.975, axis=0)

    return pd.DataFrame(
        {
            "pt": pt_values.astype(float),
            "delta_nb": delta_point.astype(float),
            "ci_low": ci_low.astype(float),
            "ci_high": ci_high.astype(float),
        }
    )

def plot_decision_curve(
    thresholds: np.ndarray,
    curves: Dict[str, np.ndarray],
    save_path: Path,
    *,
    title: str,
) -> None:
    fig, ax = plt.subplots(figsize=(9, 6))
    for name, nb in curves.items():
        ax.plot(thresholds, nb, label=name, linewidth=2)

    ax.axhline(0.0, linestyle="--", linewidth=1)
    ax.set_xlim(float(thresholds.min()), float(thresholds.max()))
    ax.set_xlabel("Threshold probability", fontsize=12)
    ax.set_ylabel("Net benefit", fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend(loc="best", fontsize=10)
    plt.tight_layout()
    _ensure_dir(save_path.parent)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

# Case difficulty stratification
def bootstrap_delta_accuracy_ci(
    *,
    y_true: np.ndarray,
    y_pred_model: np.ndarray,
    y_pred_reader: np.ndarray,
    B: int,
    seed: int,
) -> Tuple[float, float, float]:
    """Paired bootstrap CI for ΔAcc = Acc_model - Acc_reader."""
    rng = np.random.default_rng(seed)
    yt = np.asarray(y_true, dtype=int)
    pm = np.asarray(y_pred_model, dtype=int)
    prf = np.asarray(y_pred_reader, dtype=float)
    keep = np.isfinite(prf)
    yt = yt[keep]
    pm = pm[keep]
    pr = prf[keep].astype(int)

    n = int(len(yt))
    if n == 0:
        return float("nan"), float("nan"), float("nan")

    idx = np.arange(n)
    point = float((pm == yt).mean() - (pr == yt).mean())

    boot = np.empty(B, dtype=float)
    for b in range(B):
        samp = rng.choice(idx, size=n, replace=True)
        boot[b] = float((pm[samp] == yt[samp]).mean() - (pr[samp] == yt[samp]).mean())

    lo, hi = np.quantile(boot, [0.025, 0.975]).tolist()
    return float(point), float(lo), float(hi)

def case_difficulty_stratification(
    *,
    y_true: np.ndarray,
    y_pred_model: np.ndarray,
    rad1: np.ndarray,
    rad2: np.ndarray,
    rad3: np.ndarray,
    B: int,
    seed: int,
) -> pd.DataFrame:
    """Stratify cases by radiologist correctness and compute accuracies + ΔAcc CIs.

    Strata (nested):
      - Consensus easy: all 3 radiologists correct
      - Consensus hard: >=1 radiologist incorrect
      - Majority hard: >=2 radiologists incorrect
    """
    yt = np.asarray(y_true, dtype=int)
    pm = np.asarray(y_pred_model, dtype=int)
    r1f = np.asarray(rad1, dtype=float)
    r2f = np.asarray(rad2, dtype=float)
    r3f = np.asarray(rad3, dtype=float)

    keep3 = np.isfinite(r1f) & np.isfinite(r2f) & np.isfinite(r3f)
    yt3 = yt[keep3]
    pm3 = pm[keep3]
    r1 = r1f[keep3].astype(int)
    r2 = r2f[keep3].astype(int)
    r3 = r3f[keep3].astype(int)

    if int(len(yt3)) == 0:
        return pd.DataFrame()

    c1 = (r1 == yt3)
    c2 = (r2 == yt3)
    c3 = (r3 == yt3)
    n_wrong = (~c1).astype(int) + (~c2).astype(int) + (~c3).astype(int)

    strata: Dict[str, np.ndarray] = {
        "Consensus easy (all readers correct)": (n_wrong == 0),
        "Consensus hard (≥1 reader incorrect)": (n_wrong >= 1),
        "Majority hard (≥2 readers incorrect)": (n_wrong >= 2),
    }

    rows: List[Dict[str, Any]] = []
    for sname, mask in strata.items():
        if int(mask.sum()) == 0:
            rows.append(
                {"stratum": sname, "n": 0,
                 "acc_model": float("nan"), "acc_rad1": float("nan"), "acc_rad2": float("nan"), "acc_rad3": float("nan"),
                 "delta_acc_rad1": float("nan"), "delta_acc_rad1_ci_low": float("nan"), "delta_acc_rad1_ci_high": float("nan"),
                 "delta_acc_rad2": float("nan"), "delta_acc_rad2_ci_low": float("nan"), "delta_acc_rad2_ci_high": float("nan"),
                 "delta_acc_rad3": float("nan"), "delta_acc_rad3_ci_low": float("nan"), "delta_acc_rad3_ci_high": float("nan"),
                 }
            )
            continue

        yt_s = yt3[mask]
        pm_s = pm3[mask]
        r1_s = r1[mask]
        r2_s = r2[mask]
        r3_s = r3[mask]

        acc_m = float((pm_s == yt_s).mean())
        acc_r1 = float((r1_s == yt_s).mean())
        acc_r2 = float((r2_s == yt_s).mean())
        acc_r3 = float((r3_s == yt_s).mean())

        d1, lo1, hi1 = bootstrap_delta_accuracy_ci(y_true=yt_s, y_pred_model=pm_s, y_pred_reader=r1_s, B=B, seed=seed + 11)
        d2, lo2, hi2 = bootstrap_delta_accuracy_ci(y_true=yt_s, y_pred_model=pm_s, y_pred_reader=r2_s, B=B, seed=seed + 22)
        d3, lo3, hi3 = bootstrap_delta_accuracy_ci(y_true=yt_s, y_pred_model=pm_s, y_pred_reader=r3_s, B=B, seed=seed + 33)

        rows.append(
            {
                "stratum": sname,
                "n": int(len(yt_s)),
                "acc_model": acc_m,
                "acc_rad1": acc_r1,
                "acc_rad2": acc_r2,
                "acc_rad3": acc_r3,
                "delta_acc_rad1": d1,
                "delta_acc_rad1_ci_low": lo1,
                "delta_acc_rad1_ci_high": hi1,
                "delta_acc_rad2": d2,
                "delta_acc_rad2_ci_low": lo2,
                "delta_acc_rad2_ci_high": hi2,
                "delta_acc_rad3": d3,
                "delta_acc_rad3_ci_low": lo3,
                "delta_acc_rad3_ci_high": hi3,
            }
        )

    return pd.DataFrame(rows)

def plot_stratified_accuracy(
    df_strata: pd.DataFrame,
    save_path: Path,
    *,
    title: str,
) -> None:
    """Grouped bar plot for accuracies by difficulty stratum."""
    if df_strata.empty:
        return
    strata = df_strata["stratum"].tolist()
    x = np.arange(len(strata))
    width = 0.18

    acc_model = df_strata["acc_model"].to_numpy(dtype=float)
    acc_r1 = df_strata["acc_rad1"].to_numpy(dtype=float)
    acc_r2 = df_strata["acc_rad2"].to_numpy(dtype=float)
    acc_r3 = df_strata["acc_rad3"].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - 1.5 * width, acc_model, width, label="Model")
    ax.bar(x - 0.5 * width, acc_r1, width, label="Rad-1")
    ax.bar(x + 0.5 * width, acc_r2, width, label="Rad-2")
    ax.bar(x + 1.5 * width, acc_r3, width, label="Rad-3")

    ax.set_xticks(x)
    ax.set_xticklabels(strata, rotation=15, ha="right", fontsize=10)
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Accuracy", fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend(loc="best", fontsize=10)
    plt.tight_layout()
    _ensure_dir(save_path.parent)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

# Main
def run_vs_radiologists_lateral(h: Dict) -> None:
    _ensure_dir(OUT_ROOT)

    # 1) Inference ONCE (exact match)
    LOG.info("Running model inference (exact loop) ...")
    df_pred = infer_exact(h=h, ckpt_path=CKPT_PATH)

    y_true_all = df_pred["y_true"].to_numpy(dtype=int)
    y_pred_all = df_pred["model_pred"].to_numpy(dtype=int)
    y_prob_all = df_pred["model_prob"].to_numpy(dtype=float)

    m_all = _metrics_from_outputs(y_true_all, y_pred_all, y_prob_all)
    LOG.info("[Sanity] EXACT inference metrics (full test_loader):")
    LOG.info(json.dumps(m_all, indent=2))

    # 2) Load radiologist table (LAT only) and merge robustly
    LOG.info("Loading radiologist CSV and matching by filename ...")
    rad = pd.read_csv(RAD_CSV)

    # Accept either 'original' or common alternatives (defensive)
    if "original" not in rad.columns:
        for alt in ["Image", "image", "filename", "file"]:
            if alt in rad.columns:
                rad = rad.rename(columns={alt: "original"})
                break
    if "original" not in rad.columns:
        raise KeyError("Radiologist CSV is missing an 'original' (or equivalent) filename column.")

    need_cols = [
        "original", "Type", "MC_status",
        "alopezca_prediction", "lamorenog_prediction", "rdmontoyac_prediction",
    ]
    missing = [c for c in need_cols if c not in rad.columns]
    if missing:
        raise KeyError(f"Missing required columns in RAD CSV: {missing}")

    rad = rad[need_cols].copy()
    rad["original"] = rad["original"].astype(str).map(_basename)
    rad = rad.loc[rad["original"].map(_is_lateral_lat)].copy()

    # Build merge keys (case-insensitive)
    rad["orig_key"] = rad["original"].str.lower()
    df_pred["original"] = df_pred["original"].astype(str).map(_basename)
    df_pred = df_pred.loc[df_pred["original"].map(_is_lateral_lat)].copy()
    df_pred["orig_key"] = df_pred["original"].str.lower()

    # Deduplicate RAD rows if any (keep first)
    if rad["orig_key"].duplicated().any():
        ndup = int(rad["orig_key"].duplicated().sum())
        LOG.warning(f"[QC] RAD CSV has {ndup} duplicated filenames after lowercasing; keeping first occurrence.")
        rad = rad.drop_duplicates(subset=["orig_key"], keep="first")

    df = df_pred.merge(rad, on="orig_key", how="left", suffixes=("", "_rad"), validate="one_to_one")
    # Keep a canonical 'original' name from prediction side (already basename)
    df["original"] = df["original"].astype(str)

    if df["Type"].isna().any():
        missing_names = df.loc[df["Type"].isna(), "original"].tolist()[:25]
        raise RuntimeError(
            "Some test images could not be matched to RAD_CSV by filename. "
            f"First missing examples: {missing_names}"
        )

    # Reader calls -> {0,1}
    df["rad1"] = df["alopezca_prediction"].map(_to_bin_pred)
    df["rad2"] = df["lamorenog_prediction"].map(_to_bin_pred)
    df["rad3"] = df["rdmontoyac_prediction"].map(_to_bin_pred)

    CLASS_LABELS = ["Normal", "TB"]
    CMAPS = {"Model": "Greens", "Rad-1": "Reds", "Rad-2": "Blues", "Rad-3": "YlOrBr"}

    # Persist merged table once (useful for debugging/repro)
    df.to_csv(OUT_ROOT / "predictions_plus_readers_full.csv", index=False)

    cohorts = ["All", "Confirmed", "Unconfirmed"]
    for cohort in cohorts:
        cdir = OUT_ROOT / cohort
        for sub in ["confusion", "roc", "metrics", "metrics_ci", "stats", "fig_disagreement", "fig_agreement", "error_analysis"]:
            _ensure_dir(cdir / sub)

        dfc = build_cohort(df, cohort)

        y_true = dfc["y_true"].to_numpy(dtype=int)
        y_prob = dfc["model_prob"].to_numpy(dtype=float)
        y_pred_model = dfc["model_pred"].to_numpy(dtype=int)

        methods: List[Tuple[str, np.ndarray, Optional[np.ndarray]]] = [
            ("Model", y_pred_model, y_prob),
            ("Rad-1", dfc["rad1"].to_numpy(dtype=float), None),
            ("Rad-2", dfc["rad2"].to_numpy(dtype=float), None),
            ("Rad-3", dfc["rad3"].to_numpy(dtype=float), None),
        ]

        # Confusions + metrics point estimates
        rows: List[Dict[str, Any]] = []
        method_arrays: Dict[str, Dict[str, Any]] = {}

        for name, yp_raw, yprob_opt in methods:
            if yp_raw.dtype.kind == "f":
                keep = ~np.isnan(yp_raw)
                yt = y_true[keep]
                yp = yp_raw[keep].astype(int)
                yprob = None
            else:
                yt = y_true
                yp = yp_raw.astype(int)
                yprob = yprob_opt

            cm_loc = _confusion_predrows_truecols(yt, yp)
            met = _metrics_from_outputs(yt, yp, yprob)
            met["method"] = name
            met["cohort"] = cohort
            rows.append(met)

            method_arrays[name] = {"y_true": yt, "y_pred": yp, "y_prob": yprob}

            plot_confusion_pred_true(
                cm_loc,
                CLASS_LABELS,
                save_path=cdir / "confusion" / f"confusion_{name.replace(' ', '_')}.png",
            )

        metrics_df = pd.DataFrame(rows).sort_values(["cohort", "method"])
        metrics_df.to_csv(cdir / "metrics" / "metrics_summary.csv", index=False)

        # ROC: model curve + 3 radiologist points
        model_auc = float(roc_auc_score(y_true, y_prob)) if _has_both_classes(y_true) else float('nan')
        model_J = float(_youden_j_from_preds(y_true, y_pred_model))

        TNm, FPm, FNm, TPm = _counts_from_y(y_true, y_pred_model)
        model_fpr = _safe_div(FPm, FPm + TNm)
        model_tpr = _safe_div(TPm, TPm + FNm)

        rad_points: Dict[str, Tuple[float, float, float]] = {}
        for rname, col in [("Rad-1", "rad1"), ("Rad-2", "rad2"), ("Rad-3", "rad3")]:
            yp_r_raw = dfc[col].to_numpy(dtype=float)
            keep = ~np.isnan(yp_r_raw)
            yt = y_true[keep]
            yr = yp_r_raw[keep].astype(int)

            TN, FP, FN, TP = _counts_from_y(yt, yr)
            fpr_r = _safe_div(FP, FP + TN)
            tpr_r = _safe_div(TP, TP + FN)
            J = float(tpr_r + (1.0 - fpr_r) - 1.0)
            rad_points[rname] = (float(fpr_r), float(tpr_r), float(J))
        if _has_both_classes(y_true):
            plot_roc_model_plus_rads(
            y_true=y_true,
            y_prob=y_prob,
            rad_points=rad_points,
            model_J=model_J,
            model_auc=model_auc,
            model_point=(float(model_fpr), float(model_tpr)),
            save_path=cdir / "roc" / "roc_with_readers.png",
        )
        else:
            LOG.warning(f"[{cohort}] ROC curve skipped: only one class present in y_true for this cohort.")

        # Bootstrap 95% CI for each metric (Model + each reader)
        ci_rows = []
        for name in ["Model", "Rad-1", "Rad-2", "Rad-3"]:
            arr = method_arrays[name]
            ci_tbl = bootstrap_metrics_ci(
                y_true=arr["y_true"],
                y_pred=arr["y_pred"],
                y_prob=arr["y_prob"],
                B=BOOT.B_metrics_ci,
                seed=BOOT.seed,
            )
            ci_tbl.insert(0, "method", name)
            ci_tbl.insert(0, "cohort", cohort)
            ci_rows.append(ci_tbl)

        metrics_ci_df = pd.concat(ci_rows, axis=0, ignore_index=True)
        metrics_ci_df.to_csv(cdir / "metrics_ci" / "metrics_ci.csv", index=False)

        # Paired stats (McNemar exact + Holm + bootstrap ΔJ)
        stats_rows: List[Dict[str, Any]] = []
        p_overall: Dict[str, float] = {}
        p_sens: Dict[str, float] = {}

        ym = y_pred_model.astype(int)

        for name, col in [("Rad-1", "rad1"), ("Rad-2", "rad2"), ("Rad-3", "rad3")]:
            yp_r_raw = dfc[col].to_numpy(dtype=float)
            keep = ~np.isnan(yp_r_raw)

            yt = y_true[keep]
            ym_k = ym[keep]
            yr = yp_r_raw[keep].astype(int)

            # Overall McNemar on correctness
            m_correct = (ym_k == yt).astype(int)
            r_correct = (yr == yt).astype(int)
            b = int(((m_correct == 1) & (r_correct == 0)).sum())
            c = int(((m_correct == 0) & (r_correct == 1)).sum())
            p_mc = _mcnemar_exact_pvalue(b, c)
            p_overall[name] = p_mc

            # TB-positive McNemar (sensitivity focus)
            pos = (yt == 1)
            if int(pos.sum()) > 0:
                ym_pos = ym_k[pos]
                yr_pos = yr[pos]
                m_hit = (ym_pos == 1).astype(int)
                r_hit = (yr_pos == 1).astype(int)
                b2 = int(((m_hit == 1) & (r_hit == 0)).sum())
                c2 = int(((m_hit == 0) & (r_hit == 1)).sum())
                p_mc_sens = _mcnemar_exact_pvalue(b2, c2)
            else:
                b2 = c2 = 0
                p_mc_sens = float("nan")
            p_sens[name] = p_mc_sens

            deltaJ, ci_lo, ci_hi = paired_bootstrap_delta_J(
                y_true=yt,
                y_pred_model=ym_k,
                y_pred_reader=yr,
                B=BOOT.B_deltaJ,
                seed=BOOT.seed,
            )

            stats_rows.append({
                "cohort": cohort,
                "reader": name,
                "b_overall": b, "c_overall": c,
                "mcnemar_p_overall": p_mc,
                "b_sens_tbpos": b2, "c_sens_tbpos": c2,
                "mcnemar_p_sens_tbpos": p_mc_sens,
                "delta_YoudenJ": deltaJ,
                "deltaJ_CI95_low": ci_lo,
                "deltaJ_CI95_high": ci_hi,
            })

        adj_overall = _holm_bonferroni(p_overall)
        adj_sens = _holm_bonferroni(p_sens)
        for r in stats_rows:
            nm = r["reader"]
            r["mcnemar_p_overall_holm"] = adj_overall.get(nm, float("nan"))
            r["mcnemar_p_sens_tbpos_holm"] = adj_sens.get(nm, float("nan"))

        paired_df = pd.DataFrame(stats_rows).sort_values(["cohort", "reader"])
        paired_df.to_csv(cdir / "stats" / "paired_stats.csv", index=False)

        # TB+ discordance squares
        for rname, col in [("Rad-1", "rad1"), ("Rad-2", "rad2"), ("Rad-3", "rad3")]:
            yp_r_raw = dfc[col].to_numpy(dtype=float)
            keep = ~np.isnan(yp_r_raw)
            yt = y_true[keep]
            yr = yp_r_raw[keep].astype(int)
            ym_k = ym[keep]

            tbpos = (yt == 1)
            m_hit = (ym_k[tbpos] == 1)
            r_hit = (yr[tbpos] == 1)

            plot_tbpos_discordance_square(
                m_hit=m_hit,
                r_hit=r_hit,
                save_path=cdir / "fig_disagreement" / f"tbpos_discordance_{rname}.png",
                reader_name=rname,
                cohort=cohort,
                cmap=DISCORDANCE_CMAPS.get(cohort, "Greys"),
            )

        # UpSet-like intersections (all 3 readers present)
        r1f = dfc["rad1"].to_numpy(dtype=float)
        r2f = dfc["rad2"].to_numpy(dtype=float)
        r3f = dfc["rad3"].to_numpy(dtype=float)
        keep3 = (~np.isnan(r1f)) & (~np.isnan(r2f)) & (~np.isnan(r3f))

        yt3 = y_true[keep3]
        m3 = ym[keep3]
        r1 = r1f[keep3].astype(int)
        r2 = r2f[keep3].astype(int)
        r3 = r3f[keep3].astype(int)

        set_names = ["Model", "Rad-1", "Rad-2", "Rad-3"]
        set_colors = ["green", "red", "blue", "saddlebrown"]

        bits_all = np.stack([m3, r1, r2, r3], axis=1)
        tbl_all = plot_upset_like(
            bits=bits_all,
            set_names=set_names,
            set_colors=set_colors,
            save_path=cdir / "fig_agreement" / "upset_allcases.png",
            caption=f"{cohort} — intersections (all cases; all 3 readers present)",
        )
        tbl_all.to_csv(cdir / "fig_agreement" / "upset_allcases_table.csv", index=False)

        tbpos3 = (yt3 == 1)
        bits_tb = np.stack([m3[tbpos3], r1[tbpos3], r2[tbpos3], r3[tbpos3]], axis=1)
        tbl_tb = plot_upset_like(
            bits=bits_tb,
            set_names=set_names,
            set_colors=set_colors,
            save_path=cdir / "fig_agreement" / "upset_TBpos_only.png",
            caption=f"{cohort} — intersections (TB-positive only; all 3 readers present)",
        )
        tbl_tb.to_csv(cdir / "fig_agreement" / "upset_TBpos_only_table.csv", index=False)

        # Reader agreement stats
        agree = reader_agreement_summary(r1f, r2f, r3f)
        pd.DataFrame([{"cohort": cohort, **agree}]).to_csv(
            cdir / "fig_agreement" / "reader_agreement_stats.csv", index=False
        )

        # Error analysis
        err_df = error_analysis(
            y_true=y_true,
            model_pred=ym,
            rad1=r1f,
            rad2=r2f,
            rad3=r3f,
        )
        err_df.to_csv(cdir / "error_analysis" / "error_analysis.csv", index=False)
        plot_fp_controls_bar(
            err_df=err_df,
            save_path=cdir / "error_analysis" / "fp_controls_bar.png",
            caption=f"{cohort}",
        )

        # Agreement metrics (κ)
        exp1_dir = cdir / "agreement_exp1"
        _ensure_dir(exp1_dir)

        exp1_long, exp1_mat = agreement_metrics_exp1(
            y_true=y_true,
            model_pred=y_pred_model,
            rad1=dfc["rad1"].to_numpy(dtype=float),
            rad2=dfc["rad2"].to_numpy(dtype=float),
            rad3=dfc["rad3"].to_numpy(dtype=float),
        )
        exp1_long.to_csv(exp1_dir / "agreement_metrics_long.csv", index=False)
        # save Fleiss-only subset (radiologists-only and Model+3 radiologists)
        fleiss_df = exp1_long.loc[exp1_long["pair"].str.contains("Fleiss", na=False)].copy()
        fleiss_df.to_csv(exp1_dir / "fleiss_kappa_summary.csv", index=False)
        exp1_mat.to_csv(exp1_dir / "kappa_matrix.csv", index=True)
        plot_kappa_heatmap(
            exp1_mat,
            save_path=exp1_dir / "kappa_matrix_heatmap.png",
            title=f"{cohort}: Pairwise Cohen's κ (Truth/Model/Readers)",
            cmap="Greens", #Blues, Greens, coolwarm, etc
        )

        # Decision Curve Analysis (Net Benefit)
        dca_dir = cdir / "dca"
        _ensure_dir(dca_dir)

        # Threshold grid for curves (avoid {0,1} to prevent division by zero)
        dca_grid = np.linspace(0.01, 0.99, 99, dtype=float)
        # Screening-oriented thresholds for table output (edit if needed)
        dca_table_pts = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30], dtype=float)

        nb_model = decision_curve_from_prob(y_true, y_prob, dca_grid)
        nb_r1 = decision_curve_from_pred(y_true, dfc["rad1"].to_numpy(dtype=float), dca_grid)
        nb_r2 = decision_curve_from_pred(y_true, dfc["rad2"].to_numpy(dtype=float), dca_grid)
        nb_r3 = decision_curve_from_pred(y_true, dfc["rad3"].to_numpy(dtype=float), dca_grid)

        # Treat-all / treat-none references
        nb_all = decision_curve_from_pred(y_true, np.ones_like(y_true, dtype=int), dca_grid)
        nb_none = np.zeros_like(dca_grid, dtype=float)

        dca_curve_df = pd.DataFrame(
            {
                "pt": dca_grid,
                "NB_Model": nb_model,
                "NB_Rad1": nb_r1,
                "NB_Rad2": nb_r2,
                "NB_Rad3": nb_r3,
                "NB_TreatAll": nb_all,
                "NB_TreatNone": nb_none,
            }
        )
        dca_curve_df.to_csv(dca_dir / "decision_curve.csv", index=False)

        plot_decision_curve(
            dca_grid,
            curves={
                "Model": nb_model,
                "Rad-1": nb_r1,
                "Rad-2": nb_r2,
                "Rad-3": nb_r3,
                "Treat all": nb_all,
                "Treat none": nb_none,
            },
            save_path=dca_dir / "decision_curve.png",
            title=f"{cohort}: Decision Curve Analysis (Net Benefit)",
        )

        # Table + paired bootstrap ΔNB(pt) CIs vs each reader
        nb_model_tbl = decision_curve_from_prob(y_true, y_prob, dca_table_pts)
        nb_r1_tbl = decision_curve_from_pred(y_true, dfc["rad1"].to_numpy(dtype=float), dca_table_pts)
        nb_r2_tbl = decision_curve_from_pred(y_true, dfc["rad2"].to_numpy(dtype=float), dca_table_pts)
        nb_r3_tbl = decision_curve_from_pred(y_true, dfc["rad3"].to_numpy(dtype=float), dca_table_pts)
        nb_all_tbl = decision_curve_from_pred(y_true, np.ones_like(y_true, dtype=int), dca_table_pts)

        d_r1 = bootstrap_delta_net_benefit_ci(
            y_true=y_true, y_prob_model=y_prob, y_pred_reader=dfc["rad1"].to_numpy(dtype=float),
            pt_values=dca_table_pts, B=BOOT.B_deltaJ, seed=BOOT.seed + 501
        )
        d_r2 = bootstrap_delta_net_benefit_ci(
            y_true=y_true, y_prob_model=y_prob, y_pred_reader=dfc["rad2"].to_numpy(dtype=float),
            pt_values=dca_table_pts, B=BOOT.B_deltaJ, seed=BOOT.seed + 502
        )
        d_r3 = bootstrap_delta_net_benefit_ci(
            y_true=y_true, y_prob_model=y_prob, y_pred_reader=dfc["rad3"].to_numpy(dtype=float),
            pt_values=dca_table_pts, B=BOOT.B_deltaJ, seed=BOOT.seed + 503
        )

        dca_table = pd.DataFrame(
            {
                "pt": dca_table_pts,
                "NB_Model": nb_model_tbl,
                "NB_Rad1": nb_r1_tbl,
                "NB_Rad2": nb_r2_tbl,
                "NB_Rad3": nb_r3_tbl,
                "NB_TreatAll": nb_all_tbl,
                "DeltaNB_ModelMinusRad1": d_r1["delta_nb"].to_numpy(dtype=float),
                "DeltaNB_ModelMinusRad1_CI_low": d_r1["ci_low"].to_numpy(dtype=float),
                "DeltaNB_ModelMinusRad1_CI_high": d_r1["ci_high"].to_numpy(dtype=float),
                "DeltaNB_ModelMinusRad2": d_r2["delta_nb"].to_numpy(dtype=float),
                "DeltaNB_ModelMinusRad2_CI_low": d_r2["ci_low"].to_numpy(dtype=float),
                "DeltaNB_ModelMinusRad2_CI_high": d_r2["ci_high"].to_numpy(dtype=float),
                "DeltaNB_ModelMinusRad3": d_r3["delta_nb"].to_numpy(dtype=float),
                "DeltaNB_ModelMinusRad3_CI_low": d_r3["ci_low"].to_numpy(dtype=float),
                "DeltaNB_ModelMinusRad3_CI_high": d_r3["ci_high"].to_numpy(dtype=float),
            }
        )
        dca_table.to_csv(dca_dir / "net_benefit_table.csv", index=False)

        # Case difficulty stratification
        diff_dir = cdir / "difficulty"
        _ensure_dir(diff_dir)

        df_diff = case_difficulty_stratification(
            y_true=y_true,
            y_pred_model=y_pred_model,
            rad1=dfc["rad1"].to_numpy(dtype=float),
            rad2=dfc["rad2"].to_numpy(dtype=float),
            rad3=dfc["rad3"].to_numpy(dtype=float),
            B=BOOT.B_deltaJ,
            seed=BOOT.seed + 701,
        )
        df_diff.to_csv(diff_dir / "accuracy_by_difficulty.csv", index=False)
        plot_stratified_accuracy(
            df_diff,
            save_path=diff_dir / "stratified_accuracy.png",
            title=f"{cohort}: Accuracy by Case Difficulty",
        )

    LOG.info(f"✅ Done. Outputs saved under: {OUT_ROOT}")

# RUN
run_vs_radiologists_lateral(h)

In [ ]:
# CAM visualization
# - Saves: masks/heatmaps/(optional)contours/bboxes + index.csv under:
#     <output_dir>/<model>/<cam>_fold<k>__thrX.XX/test/...

# Logging
LOG = logging.getLogger("cam_test")
if not LOG.handlers:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

CHOSEN_FOLD: int = 1
CHOSEN_CKPT: str = "/path/to/your/model.ckpt"

CAM_METHOD: str = "eigencam"           # "eigencam" | "gradcam" etc.
ABNORMAL_CLASS_INDEX: int = 1          # positive class = normal_not/TB
HEATMAP_ALPHA: float = 0.60
SAVE_CONTOURS: bool = True

THRESHOLD_CAM: float = 0.70           # SINGLE threshold (applied once to normalized CAM)
PRESERVE_INTENSITY: bool = True       # True -> keep intensities above thr; False -> binary mask

CONTOUR_COLOR: Tuple[int, int, int] = (0, 0, 255)  # BGR
CONTOUR_THICKNESS: int = 3
CONTOUR_LINE_TYPE: int = cv2.LINE_AA
IMAGE_EXTS: Tuple[str, ...] = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp")

# Preprocessing
def build_preprocess(img_size: int) -> transforms.Compose:
    return transforms.Compose(
        [
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ]
    )

def preprocess_image(orig_rgb01: np.ndarray, preprocess: transforms.Compose, device: torch.device) -> torch.Tensor:
    img_u8 = (np.clip(orig_rgb01, 0.0, 1.0) * 255.0).astype(np.uint8)
    return preprocess(img_u8).unsqueeze(0).to(device)

def _amp_autocast(device: torch.device):
    if device.type != "cuda":
        return torch.cuda.amp.autocast(enabled=False)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.cuda.amp.autocast(enabled=True, dtype=dtype)

# Target layer selection (dpn68)
def _kernel_tuple(m: nn.Conv2d) -> Tuple[int, int]:
    ks = m.kernel_size
    return (int(ks[0]), int(ks[1])) if isinstance(ks, tuple) else (int(ks), int(ks))

def _last_conv_kgt1(module: nn.Module) -> Optional[nn.Conv2d]:
    last_any: Optional[nn.Conv2d] = None
    last_kgt1: Optional[nn.Conv2d] = None
    for m in module.modules():
        if isinstance(m, nn.Conv2d):
            last_any = m
            if max(_kernel_tuple(m)) > 1:
                last_kgt1 = m
    return last_kgt1 if last_kgt1 is not None else last_any

def select_target_layer_dpn68_new(model: nn.Module) -> nn.Conv2d:
    if hasattr(model, "backbone") and isinstance(getattr(model, "backbone"), nn.Module):
        tl = _last_conv_kgt1(getattr(model, "backbone"))
        if isinstance(tl, nn.Conv2d):
            return tl
    tl = _last_conv_kgt1(model)
    if isinstance(tl, nn.Conv2d):
        return tl
    raise RuntimeError("No Conv2d layer found for CAM (unexpected for dpn68_new CNN).")

def _find_module_name(root: nn.Module, target: nn.Module) -> str:
    for name, m in root.named_modules():
        if m is target:
            return name or "<root>"
    return "<unnamed>"

# Model loading
def load_model_auto_strict(ckpt_path: Union[str, Path], h: Dict[str, object], device: torch.device) -> nn.Module:
    ckpt_path = str(ckpt_path)
    try:
        model = TBModel.load_from_checkpoint(  # type: ignore[name-defined]
            ckpt_path, h=h, strict=True, map_location=device
        )
        LOG.info("[LOAD] strict=True worked.")
    except Exception as e:
        LOG.warning(f"[LOAD] strict=True failed: {e} | retrying strict=False.")
        model = TBModel.load_from_checkpoint(  # type: ignore[name-defined]
            ckpt_path, h=h, strict=False, map_location=device
        )
        LOG.info("[LOAD] strict=False used.")
    return model.to(device).eval()

# TEST discovery (normal_not only)
def collect_normal_not_images(split_root: Path) -> List[Path]:
    preferred = split_root / "normal_not"
    images: List[Path] = []

    if preferred.exists():
        for p in preferred.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                images.append(p)
    else:
        for p in split_root.rglob("*"):
            if not (p.is_file() and p.suffix.lower() in IMAGE_EXTS):
                continue
            if "normal_not" in {part.lower() for part in p.parts}:
                images.append(p)

    images = sorted(images, key=lambda x: x.name.lower())
    if not images:
        raise RuntimeError(f"No normal_not images found under: {split_root}")
    return images

# CAM post-processing (SINGLE threshold)
def normalize_cam_to_unit(cam_map: np.ndarray) -> np.ndarray:
    cam_map = cam_map.astype(np.float32, copy=False)
    mmin, mmax = float(np.min(cam_map)), float(np.max(cam_map))
    if mmax <= mmin:
        return np.zeros_like(cam_map, dtype=np.float32)
    return (cam_map - mmin) / (mmax - mmin)

def threshold_cam_once(cam01: np.ndarray, thr: float, preserve_intensity: bool = True) -> np.ndarray:
    gate = (cam01 >= float(thr)).astype(np.float32)
    return (cam01 * gate) if preserve_intensity else gate

# Main
def run_cam_dpn68(
    *,
    h: Dict[str, object],
    chosen_fold: int,
    chosen_ckpt_name: str,
    method: str = "eigencam",
    abnormal_class_index: int = 1,
    heatmap_alpha: float = 0.6,
    save_contours: bool = True,
    threshold_cam: float = 0.7,
    preserve_intensity: bool = True,
) -> Path:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_id = str(h["model"])
    img_size = int(h["img_size"])
    preprocess = build_preprocess(img_size)
    ckpt_dir = Path(str(h["output_dir"])) / f"fold_{chosen_fold}" / model_id
    ckpt_path = ckpt_dir / chosen_ckpt_name
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    model = load_model_auto_strict(ckpt_path, h=h, device=device)

    target_layer = select_target_layer_dpn68_new(model)
    LOG.info(f"[CAM] target_layer={_find_module_name(model, target_layer)} | kernel={_kernel_tuple(target_layer)}")

    mname = method.strip().lower()
    if mname not in {"eigencam", "gradcam"}:
        raise ValueError("CAM_METHOD must be one of: 'eigencam', 'gradcam'")
    cam_obj = EigenCAM(model=model, target_layers=[target_layer]) if mname == "eigencam" else GradCAM(model=model, target_layers=[target_layer])
    cam_obj.batch_size = 1

    out_root = (
        Path(str(h["output_dir"]))
        / model_id
        / f"{mname}_fold{chosen_fold}__thr{float(threshold_cam):.2f}"
        / "test"
    )
    out_root.mkdir(parents=True, exist_ok=True)

    (out_root / "masks").mkdir(parents=True, exist_ok=True)
    (out_root / "heatmaps").mkdir(parents=True, exist_ok=True)
    if save_contours:
        (out_root / "contours").mkdir(parents=True, exist_ok=True)
        (out_root / "bboxes").mkdir(parents=True, exist_ok=True)

    test_root = Path(str(h["test_dir"]))
    img_paths = collect_normal_not_images(test_root)
    LOG.info(f"[TEST] {len(img_paths)} normal_not images @ {test_root}")
    LOG.info(f"[THR] single CAM threshold: thr={threshold_cam:.3f} (preserve_intensity={preserve_intensity})")
    LOG.info(f"[OUT] {out_root}")

    records: List[Dict[str, object]] = []

    with torch.inference_mode():
        for idx, img_fp in enumerate(img_paths, start=1):
            bgr = cv2.imread(str(img_fp), cv2.IMREAD_COLOR)
            if bgr is None:
                LOG.warning(f"[WARN] cannot read: {img_fp}")
                continue

            H0, W0 = bgr.shape[:2]
            rgb01 = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            inp = preprocess_image(rgb01, preprocess, device)

            # model probability (QC)
            with _amp_autocast(device):
                logits = model(inp)
            logits = logits.float()
            prob_tb = float(torch.softmax(logits, dim=1)[0, int(abnormal_class_index)].detach().cpu().item())
            pred = int(torch.argmax(logits, dim=1)[0].detach().cpu().item())

            # CAM
            if mname == "eigencam":
                cam_raw = cam_obj(input_tensor=inp)[0]
            else:
                cam_raw = cam_obj(
                    input_tensor=inp,
                    targets=[ClassifierOutputTarget(int(abnormal_class_index))],
                    aug_smooth=True,
                    eigen_smooth=True,
                )[0]

            cam01 = normalize_cam_to_unit(cam_raw)
            cam_thr = threshold_cam_once(cam01, threshold_cam, preserve_intensity=preserve_intensity)

            cam_u8_small = np.uint8(np.clip(cam_thr, 0.0, 1.0) * 255.0)
            cam_u8 = cv2.resize(cam_u8_small, (W0, H0), interpolation=cv2.INTER_LINEAR)

            tag = f"{img_fp.stem}__{model_id}__fold{chosen_fold}__{mname}"

            mask_fp = out_root / "masks" / f"{tag}__mask.png"
            cv2.imwrite(str(mask_fp), cam_u8)

            heat = cv2.applyColorMap(cam_u8, cv2.COLORMAP_HOT)
            overlay = cv2.addWeighted(heat, float(heatmap_alpha), bgr, 1.0 - float(heatmap_alpha), 0.0)
            heat_fp = out_root / "heatmaps" / f"{tag}__heat.png"
            cv2.imwrite(str(heat_fp), overlay)

            n_contours = 0
            if save_contours:
                _, binary = cv2.threshold(cam_u8, 1, 255, cv2.THRESH_BINARY)

                kernel = np.ones((3, 3), np.uint8)
                binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)
                binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=1)

                contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                n_contours = int(len(contours))

                cont_img = bgr.copy()
                cv2.drawContours(cont_img, contours, -1, CONTOUR_COLOR, CONTOUR_THICKNESS, lineType=CONTOUR_LINE_TYPE)
                cv2.imwrite(str(out_root / "contours" / f"{tag}__contours.png"), cont_img)

                box_img = bgr.copy()
                for cnt in contours:
                    x, y, w, h_ = cv2.boundingRect(cnt)
                    cv2.rectangle(
                        box_img, (x, y), (x + w, y + h_),
                        CONTOUR_COLOR, CONTOUR_THICKNESS, lineType=CONTOUR_LINE_TYPE
                    )
                cv2.imwrite(str(out_root / "bboxes" / f"{tag}__bboxes.png"), box_img)

            records.append(
                {
                    "split": "test",
                    "filename": img_fp.name,
                    "orig_path": str(img_fp),
                    "prob_tb": prob_tb,
                    "pred": pred,
                    "cam_max": float(np.max(cam01)),
                    "cam_mean": float(np.mean(cam01)),
                    "n_contours": n_contours,
                    "mask_path": str(mask_fp),
                    "heat_path": str(heat_fp),
                }
            )

            if idx % 100 == 0 or idx == len(img_paths):
                LOG.info(f"[test] processed {idx}/{len(img_paths)}")

    idx_csv = out_root / "index.csv"
    pd.DataFrame(records).to_csv(idx_csv, index=False)
    LOG.info(f"[test] wrote index: {idx_csv}")

    if hasattr(cam_obj, "activations_and_grads"):
        try:
            cam_obj.activations_and_grads.release()
        except Exception:
            pass
    del cam_obj, model
    if device.type == "cuda":
        torch.cuda.empty_cache()

    LOG.info(f"✅ Done. Outputs saved under: {out_root}")
    return out_root

# Sanity check
def sanity_check(
    test_out_root: Union[str, Path],
    *,
    n_samples: int = 6,
    seed: int = 1337,
    show_masks: bool = True,
) -> None:
    test_out_root = Path(test_out_root)
    idx_csv = test_out_root / "index.csv"
    if not idx_csv.exists():
        LOG.warning(f"[sanity] missing index: {idx_csv}")
        return

    df = pd.read_csv(idx_csv)
    if df.empty:
        LOG.warning("[sanity] empty index.csv")
        return

    picks = df.sample(n=min(n_samples, len(df)), random_state=seed)
    LOG.info(
        f"[sanity:test] n={len(df)} | "
        f"prob_tb mean={df['prob_tb'].mean():.3f} min={df['prob_tb'].min():.3f} max={df['prob_tb'].max():.3f} | "
        f"n_contours mean={df['n_contours'].mean():.2f}"
    )

    ncols = 3 if show_masks else 2
    fig, axes = plt.subplots(len(picks), ncols, figsize=(5.5 * ncols, 4.2 * len(picks)))
    if len(picks) == 1:
        axes = np.array([axes])

    for r, row in enumerate(picks.to_dict("records")):
        orig_path = Path(row["orig_path"])
        heat_path = Path(row["heat_path"])
        mask_path = Path(row["mask_path"])

        ax0 = axes[r, 0]
        bgr = cv2.imread(str(orig_path), cv2.IMREAD_COLOR)
        if bgr is None:
            ax0.text(0.5, 0.5, "MISSING ORIGINAL", ha="center", va="center", fontsize=12)
        else:
            ax0.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        ax0.set_title(f"{orig_path.name}\nprob_tb={row['prob_tb']:.3f}, pred={int(row['pred'])}")
        ax0.axis("off")

        ax1 = axes[r, 1]
        heat = cv2.imread(str(heat_path), cv2.IMREAD_COLOR)
        if heat is None:
            ax1.text(0.5, 0.5, "MISSING HEATMAP", ha="center", va="center", fontsize=12)
        else:
            ax1.imshow(cv2.cvtColor(heat, cv2.COLOR_BGR2RGB))
        ax1.set_title("Overlay (thresholded CAM)")
        ax1.axis("off")

        if show_masks:
            ax2 = axes[r, 2]
            m = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if m is None:
                ax2.text(0.5, 0.5, "MISSING MASK", ha="center", va="center", fontsize=12)
            else:
                ax2.imshow(m, cmap="gray")
            ax2.set_title(f"Mask (thr={THRESHOLD_CAM:.2f})")
            ax2.axis("off")

    plt.tight_layout()
    plt.show()

# Run
TEST_OUT = run_cam_dpn68(
    h=h,
    chosen_fold=CHOSEN_FOLD,
    chosen_ckpt_name=CHOSEN_CKPT,
    method=CAM_METHOD,
    abnormal_class_index=ABNORMAL_CLASS_INDEX,
    heatmap_alpha=HEATMAP_ALPHA,
    save_contours=SAVE_CONTOURS,
    threshold_cam=THRESHOLD_CAM,
    preserve_intensity=PRESERVE_INTENSITY,
)
sanity_check(
    TEST_OUT,
    n_samples=6,
    seed=1337,
    show_masks=True,
)

## END OF CODE